# One aim here is building various classifiers - detailed shortly - for the GTZAN music genre collection. Each model uses a distinct network layout. Performance numbers must be shared once testing finishes. A short reflection on outcomes follows the data. Results come from working through each architecture separately.

# DataSet - The GTZAN dataset is downloaded from: https://www.kaggle.com/datasets/andradaolteanu/gtzandataset-music-genre-classification 

In [ ]:
import os
import numpy as np
try:
    import kagglehub
    import torchvision
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import transforms
    from torchvision.datasets import ImageFolder
    from torch.utils.data import DataLoader
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from sklearn.metrics import confusion_matrix, classification_report
    import seaborn as sns
except ModuleNotFoundError:
    %pip install kagglehub torch torchvision pandas matplotlib seaborn scikit-learn
    import kagglehub
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import transforms
    from torchvision.datasets import ImageFolder
    from torch.utils.data import DataLoader
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from sklearn.metrics import confusion_matrix, classification_report
    import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Downloaded then latest version of GTZAN dataset (music genre classification) from Kaggle
abs_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Absolute path:", abs_path)

# Converted to a relative path (relative to the notebook's working directory)
path = os.path.relpath(abs_path)
print("Relative path:", path)

# A set of ten categories, each holding one hundred thirty-second recordings - this makes up the full collection known as genres original. Often compared to sound’s version of MNIST, it draws attention through simplicity rather than scale. Every clip fits exactly half a minute, forming a uniform structure across all groups. Ten times a hundred pieces form the entire body, carefully split by type. Known widely by researchers, its influence grows quietly behind experiments. Though older, it remains part of many tests because access stays open and layout stays clear. Each genre acts like a branch, supporting equal weight in number and duration. Not built for flash, yet chosen often in practice. Files stay short on purpose, making processing faster without extra load. Foundational? Maybe. Still used? Definitely

# A single picture stands in for every sound recording. Classifying information can happen via systems built to mimic brain function. Since those models - convolutional ones, specifically - often rely on visuals as input, a transformation was needed. Audio clips became Mel Spectrograms, aligning them with what the network expects. This shift allows processing where it otherwise would not work.

# A pair of CSV documents holds details about sound recordings. Each track - thirty seconds in duration - appears with averaged values and spread across various measurable traits pulled from audio. In one document, these measurements come from full clips. The second uses segments cut down to three seconds, multiplying the total entries tenfold for model training purposes. Larger datasets tend to support stronger pattern recognition. Expanding sample count often improves outcome reliability.

In [ ]:
data_dir = os.path.join(path, "Data") 
print("Data directory:", data_dir)
#Reading the csv file containing the features extracted from the audio files
final_data = pd.read_csv(os.path.join(data_dir, "features_3_sec.csv"))
final_data.head()
final_data.info()
final_data.shape
final_data.dtypes

# Mandatory Step for model 1 to 4
# Function to resize all the images to 180x180 pixels.This is done as the images are loaded using torchvision.transforms.Resize. 

In [ ]:
# --- Common image transform: Resize to 180x180 + convert to tensor + normalize ---
IMG_HEIGHT = 180
IMG_WIDTH = 180
data_dir = os.path.join(path, "images_original")  # Assuming images are in 'images_original' subdirectory

def get_image_transforms(augment=False):
    """ Returns a common torchvision transform pipeline that resizes images to 180x180.
    Args:augment (bool): If True, includes data augmentation transforms (random horizontal flip, rotation) for training.If False, only resizing and normalization are applied.
    Returns: torchvision.transforms.Compose: The composed transform pipeline.
    """
    if augment:
        return transforms.Compose([
            transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


def load_image_dataset(data_dir, augment=False, batch_size=32, shuffle=True):
    """ Loads an image dataset from a directory using ImageFolder with 180x180 resizing.
    Args:data_dir (str): Path to the root directory of images (with subdirectories per class).
        augment (bool): Whether to apply data augmentation transforms.
        batch_size (int): Number of images per batch.
        shuffle (bool): Whether to shuffle the dataset.
    Returns: tuple: (DataLoader, ImageFolder dataset)
    """
    transform = get_image_transforms(augment=augment)
    dataset = ImageFolder(root=data_dir, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2)
    return dataloader, dataset


# Quick test: print the transforms
print("Training transforms (with augmentation):")
print(get_image_transforms(augment=True))

# print("\nValidation/Test transforms (no augmentation):")
# print(get_image_transforms(augment=False))

# Mandatory Step for model 1 to 4
# Function to randomly split the dataset into a training (70%), validation (20%), and test (10%) datasets using PyTorch

In [ ]:
from torch.utils.data import random_split, DataLoader

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1,batch_size=32, seed=42):
    """Randomly splits a PyTorch dataset into training, validation, and test subsets.
    Args:dataset (torch.utils.data.Dataset): The full dataset to split.
        train_ratio (float): Fraction of data for training (default: 0.7).
        val_ratio (float): Fraction of data for validation (default: 0.2).
        test_ratio (float): Fraction of data for testing (default: 0.1).
        batch_size (int): Number of samples per batch in each DataLoader.
        seed (int): Random seed for reproducibility.
    Returns:dict: A dictionary with keys 'train', 'val', 'test', each containing a dict with 'dataset' (Subset) and 'loader' (DataLoader).
    Raises: ValueError: If the ratios do not sum to 1.0.
    """
    # Validate ratios
    if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError(
            f"Ratios must sum to 1.0, got {train_ratio + val_ratio + test_ratio:.4f}"
        )

    total_size = len(dataset)
    train_size = int(total_size * train_ratio)
    val_size = int(total_size * val_ratio)
    test_size = total_size - train_size - val_size  # remainder goes to test to avoid rounding issues

    # Reproducible random split
    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

    # Create DataLoaders
    # NOTE: drop_last=True on train_loader prevents the last batch from having
    #       only 1 sample, which would cause BatchNorm to fail during training.
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,num_workers=2, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=2, drop_last=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,num_workers=2, drop_last=False)

    splits = {
        'train': {'dataset': train_dataset, 'loader': train_loader},
        'val':   {'dataset': val_dataset,   'loader': val_loader},
        'test':  {'dataset': test_dataset,  'loader': test_loader},
    }

    print(f"Dataset split (seed={seed}):")
    print(f"  Total:      {total_size:,} samples")
    print(f"  Training:   {train_size:,} samples ({train_ratio:.0%})")
    print(f"  Validation: {val_size:,} samples ({val_ratio:.0%})")
    print(f"  Test:       {test_size:,} samples ({test_ratio:.0%})")

    return splits


# --- Example usage (uncomment when dataset is loaded) ---
data_dir = path + "/Data/images_original"
full_dataset = ImageFolder(root=data_dir, transform=get_image_transforms(augment=False))
splits = split_dataset(full_dataset, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1, batch_size=32)
train_loader = splits['train']['loader']
val_loader   = splits['val']['loader']
test_loader  = splits['test']['loader']

print("split_dataset() is ready to use.")
print("Splits: 70% train | 20% validation | 10% test")

# Data VISUALIZATION: Dataset Overview — Sample Images + Split Distribution (Graphs)

In [ ]:
# ============================================================
# VISUALIZATION: Dataset Overview — Sample Images + Split Distribution
# ============================================================

fig = plt.figure(figsize=(18, 10))

# --- 1. Sample images from each genre (top grid) ---
class_names = full_dataset.classes
# Inverse normalization for display
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# Show 1 sample per genre (10 genres)
for i, class_name in enumerate(class_names):
    ax = fig.add_subplot(2, 5, i + 1)
    # Find first index of this class
    idx = next(j for j, (_, label) in enumerate(full_dataset.samples) if label == i)
    img, label = full_dataset[idx]
    img = inv_normalize(img)
    img = img.permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(class_name.capitalize(), fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('MEL Spectrogram Samples — One Per Genre (180x180)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 2. Dataset Split Pie Chart + Class Distribution Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: Train / Val / Test split
split_sizes = [len(splits['train']['dataset']),len(splits['val']['dataset']),len(splits['test']['dataset'])]
colors_pie = ['#2ecc71', '#3498db', '#e74c3c']
explode = (0.05, 0.05, 0.05)
axes[0].pie(split_sizes, labels=['Train (70%)', 'Val (20%)', 'Test (10%)'],
            autopct='%1.0f%%', colors=colors_pie, explode=explode,
            shadow=True, startangle=140, textprops={'fontsize': 12})
axes[0].set_title('Dataset Split Distribution', fontsize=14, fontweight='bold')

# Bar chart: Samples per genre
genre_counts = {}
for _, label in full_dataset.samples:
    genre = class_names[label]
    genre_counts[genre] = genre_counts.get(genre, 0) + 1

genres = list(genre_counts.keys())
counts = list(genre_counts.values())
colors_bar = plt.cm.Set3(np.linspace(0, 1, len(genres)))

bars = axes[1].bar(genres, counts, color=colors_bar, edgecolor='gray', linewidth=0.5)
axes[1].set_xlabel('Genre', fontsize=12)
axes[1].set_ylabel('Number of Samples', fontsize=12)
axes[1].set_title('Samples Per Genre', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Audio Dataset — Load Raw WAV Files & Split
# Load the 1000 `.wav` audio files from `Data/genres_original`, extract Mel-spectrogram
# features using **librosa**, build a PyTorch `TensorDataset`, and split into
# **70 / 20 / 10** train / val / test using the same `split_dataset()` function.

In [ ]:
try:
    import librosa
except ModuleNotFoundError:
    %pip install librosa
    import librosa

from torch.utils.data import TensorDataset
from sklearn.preprocessing import LabelEncoder

# ============================================================
# AUDIO DATASET — Load WAV files, extract Mel spectrograms, split
# ============================================================

audio_dir = os.path.join(path, "Data", "genres_original")
print("Audio directory:", audio_dir)
print("Genres found:", sorted(os.listdir(audio_dir)))

# --- Parameters ---
SAMPLE_RATE  = 22050        # native sample rate of GTZAN files
DURATION     = 30           # seconds per clip
N_MELS       = 128          # number of Mel bands
HOP_LENGTH   = 512          # hop length for STFT
N_FFT        = 2048         # FFT window size

# Expected time frames for a 30-second clip
EXPECTED_FRAMES = 1 + int(SAMPLE_RATE * DURATION / HOP_LENGTH)  # 1292
print(f"Mel spectrogram shape per clip: ({N_MELS}, {EXPECTED_FRAMES})")

# --- Load all audio files and extract Mel spectrograms ---
mel_features = []
labels = []
skipped = []

genres = sorted(os.listdir(audio_dir))
for genre in genres:
    genre_path = os.path.join(audio_dir, genre)
    if not os.path.isdir(genre_path):
        continue
    wav_files = sorted([f for f in os.listdir(genre_path) if f.endswith(".wav")])
    for wav_file in wav_files:
        file_path = os.path.join(genre_path, wav_file)
        try:
            # Load audio at native sample rate
            y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
            
            # Pad or truncate to exactly DURATION seconds
            expected_length = SAMPLE_RATE * DURATION
            if len(y) < expected_length:
                y = np.pad(y, (0, expected_length - len(y)))
            else:
                y = y[:expected_length]
            
            # Extract Mel spectrogram (in dB scale)
            mel = librosa.feature.melspectrogram(
                y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)  # shape: (N_MELS, EXPECTED_FRAMES)
            
            mel_features.append(mel_db)
            labels.append(genre)
        except Exception as e:
            skipped.append((wav_file, str(e)))

print(f"\nLoaded {len(mel_features)} audio files")
if skipped:
    print(f"Skipped {len(skipped)} files: {skipped}")

# --- Encode labels ---
audio_label_encoder = LabelEncoder()
y_encoded = audio_label_encoder.fit_transform(labels)  # 0..9
print(f"Genre mapping: {dict(zip(audio_label_encoder.classes_, audio_label_encoder.transform(audio_label_encoder.classes_)))}")

# --- Convert to tensors ---
# Mel spectrograms → add channel dim → (N, 1, N_MELS, FRAMES)
X_audio = np.array(mel_features)  # (N, 128, 1292)
X_tensor = torch.tensor(X_audio, dtype=torch.float32).unsqueeze(1)  # (N, 1, 128, 1292)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

print(f"\nFeature tensor shape: {X_tensor.shape}  (samples, channels, mel_bands, time_frames)")
print(f"Label tensor shape:   {y_tensor.shape}")

# --- Build TensorDataset and split using split_dataset() ---
audio_dataset = TensorDataset(X_tensor, y_tensor)

audio_splits = split_dataset(audio_dataset, train_ratio=0.7, val_ratio=0.2,
                              test_ratio=0.1, batch_size=32, seed=42)

audio_train_loader = audio_splits['train']['loader']
audio_val_loader   = audio_splits['val']['loader']
audio_test_loader  = audio_splits['test']['loader']

# --- Sanity check ---
for batch_X, batch_y in audio_train_loader:
    print(f"\nSanity check — first training batch:")
    print(f"  Mel spectrograms: {batch_X.shape}  (batch, 1, {N_MELS}, {EXPECTED_FRAMES})")
    print(f"  Labels:           {batch_y.shape}")
    print(f"  Label values:     {batch_y[:8].tolist()} ...")
    break

print(f"\nAudio loaders ready  (audio_train_loader, audio_val_loader, audio_test_loader)")
print(f"Label encoder stored as: audio_label_encoder")


# Model 1: The Simple Model - a fully connected network with two hidden layers

In [ ]:

class MusicGenreClassifier(nn.Module):
    """Fully connected network with two hidden layers for music genre classification."""

    def __init__(self, input_size=57, hidden1=256, hidden2=128, num_classes=10, dropout=0.3):
        super(MusicGenreClassifier, self).__init__()
        self.network = nn.Sequential(
            # Hidden layer 1
            nn.Linear(input_size, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Hidden layer 2
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Output layer
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# Instantiate the model
input_size=57 # -> typical number of audio features (e.g., MFCCs, chroma, spectral contrast, etc.)
num_classes=10 # -> GTZAN has 10 genres: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock
model = MusicGenreClassifier(input_size=57, hidden1=256, hidden2=128, num_classes=10)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

# VISUALIZATION: Model 1 — FC Network Architecture (Horizontal Funnel Chart)


In [ ]:
# ============================================================
# VISUALIZATION: Model 1 — FC Network Architecture (Horizontal Funnel Chart)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- 1. Layer-by-layer neuron count funnel ---
layers = ['Input\n(57)', 'Hidden 1\n(256)', 'Hidden 2\n(128)', 'Output\n(10)']
neurons = [57, 256, 128, 10]
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

bars = axes[0].barh(layers[::-1], neurons[::-1], color=colors[::-1],
                     edgecolor='white', linewidth=2, height=0.6)
for bar, n in zip(bars, neurons[::-1]):
    axes[0].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2.,
                 f'{n} neurons', va='center', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Number of Neurons', fontsize=12)
axes[0].set_title('Model 1: FC Network — Layer Sizes', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 320)

# --- 2. Parameter distribution per layer (donut chart) ---
param_counts = {}
for name, param in model.named_parameters():
    layer_name = name.split('.')[1]  # get the Sequential index
    if layer_name not in param_counts:
        param_counts[layer_name] = 0
    param_counts[layer_name] += param.numel()

# Group by meaningful names
grouped = {'Input→H1\n(Linear+BN)': 0, 'H1→H2\n(Linear+BN)': 0, 'H2→Out\n(Linear)': 0}
for key, val in param_counts.items():
    idx = int(key)
    if idx <= 1:
        list(grouped.values())  # just access
        k = list(grouped.keys())[0]
        grouped[k] += val
    elif idx <= 5:
        k = list(grouped.keys())[1]
        grouped[k] += val
    else:
        k = list(grouped.keys())[2]
        grouped[k] += val

sizes = list(grouped.values())
labels = [f'{k}\n({v:,} params)' for k, v in grouped.items()]
colors_donut = ['#2ecc71', '#f39c12', '#e74c3c']
wedges, texts, autotexts = axes[1].pie(sizes, labels=labels, autopct='%1.1f%%',
                                        colors=colors_donut, startangle=90,
                                        pctdistance=0.75, textprops={'fontsize': 10})
# Draw donut hole
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
axes[1].add_artist(centre_circle)
axes[1].set_title(f'Model 1: Parameter Distribution\n(Total: {sum(sizes):,})',
                   fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

 # Model 2: The Pattern Recognizer
 # A convolutional network based on the coursework description figure 1 & with our own choice of parameters.

In [ ]:
class MusicGenreCNN(nn.Module):
    """ Convolutional Neural Network for music genre classification.
Base architecture (from Figure 1 — NO Batch Normalisation):
        Input (3x180x180)
        → Conv1 (3→32, 3x3) + ReLU
        → Conv2 (32→64, 3x3) + ReLU
        → MaxPool (2x2)
        → Conv3 (64→128, 3x3) + ReLU
        → Conv4 (128→128, 3x3) + ReLU
        → MaxPool (2x2)
        → Flatten
        → FC1 (→256) + ReLU + Dropout
        → FC2 (→10)  [Output]
    """

    def __init__(self, num_classes=10, dropout=0.4):
        super(MusicGenreCNN, self).__init__()
        # --- Convolutional feature extractor (NO BatchNorm — matches Figure 1) ---
        self.features = nn.Sequential(
            # Conv1 + ReLU
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            # Conv2 + ReLU
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),

            # MaxPool 1
            nn.MaxPool2d(kernel_size=2, stride=2),       # 180x180 → 90x90

            # Conv3 + ReLU
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            # Conv4 + ReLU
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            # MaxPool 2
            nn.MaxPool2d(kernel_size=2, stride=2),       # 90x90 → 45x45
        )

        # --- Classifier (fully connected layers) ---
        # After features: 128 channels x 45 x 45 = 259,200
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 45 * 45, 256),               # FC1
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),                  # FC2 (Output)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# --- Instantiate the CNN model ---
cnn_model = MusicGenreCNN(num_classes=10, dropout=0.4)

# Loss function and optimizer
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)

print(cnn_model)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

# Verify output shape with a dummy input
dummy = torch.randn(1, 3, 180, 180)
out = cnn_model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")

# VISUALIZATION: Model 2 — CNN Feature Map Progression (Line + Area Chart)

In [ ]:
# ============================================================
# VISUALIZATION: Model 2 — CNN Feature Map Progression (Line + Area Chart)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Data for the CNN pipeline
stage_labels = ['Input', 'Conv1', 'Conv2', 'Pool1', 'Conv3', 'Conv4', 'Pool2', 'Flatten', 'FC1', 'FC2']
channels     = [3,       32,      64,      64,      128,     128,     128,     None,      256,   10]
spatial      = [180,     180,     180,     90,      90,      90,      45,      None,      None,  None]

# --- 1. Channel progression (step chart) ---
ch_labels = stage_labels[:7]
ch_vals = channels[:7]
axes[0].fill_between(range(len(ch_labels)), ch_vals, alpha=0.3, color='#3498db', step='mid')
axes[0].step(range(len(ch_labels)), ch_vals, where='mid', linewidth=2.5, color='#2980b9', marker='o', markersize=8)
for i, (label, val) in enumerate(zip(ch_labels, ch_vals)):
    axes[0].annotate(f'{val}', (i, val), textcoords="offset points",
                     xytext=(0, 12), ha='center', fontsize=11, fontweight='bold', color='#2c3e50')
axes[0].set_xticks(range(len(ch_labels)))
axes[0].set_xticklabels(ch_labels, rotation=30, ha='right')
axes[0].set_ylabel('Number of Channels', fontsize=12)
axes[0].set_title('Channel Depth Progression', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 160)

# --- 2. Spatial dimension reduction (waterfall) ---
sp_labels = ['Input', 'Conv1-2', 'Pool1', 'Conv3-4', 'Pool2']
sp_vals = [180, 180, 90, 90, 45]
colors_sp = ['#27ae60' if sp_vals[i] == sp_vals[i-1] or i == 0 else '#e74c3c' for i in range(len(sp_vals))]
bars = axes[1].bar(sp_labels, sp_vals, color=colors_sp, edgecolor='white', linewidth=2, width=0.6)
for bar, val in zip(bars, sp_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                 f'{val}x{val}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Spatial Size (pixels)', fontsize=12)
axes[1].set_title('Spatial Dimension Reduction', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 220)
# Legend
green_patch = mpatches.Patch(color='#27ae60', label='Preserved')
red_patch = mpatches.Patch(color='#e74c3c', label='Halved (MaxPool)')
axes[1].legend(handles=[green_patch, red_patch], loc='upper right', fontsize=10)

# --- 3. Total activations per layer (log scale) ---
total_activations = [3*180*180, 32*180*180, 64*180*180, 64*90*90,
                     128*90*90, 128*90*90, 128*45*45]
act_labels = stage_labels[:7]
axes[2].semilogy(range(len(act_labels)), total_activations, 'o-',
                 color='#8e44ad', linewidth=2.5, markersize=8)
axes[2].fill_between(range(len(act_labels)), total_activations, alpha=0.15, color='#8e44ad')
for i, val in enumerate(total_activations):
    axes[2].annotate(f'{val:,}', (i, val), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=9, rotation=20)
axes[2].set_xticks(range(len(act_labels)))
axes[2].set_xticklabels(act_labels, rotation=30, ha='right')
axes[2].set_ylabel('Total Activations (log)', fontsize=12)
axes[2].set_title('Activation Volume Through Network', fontsize=13, fontweight='bold')

plt.suptitle('Model 2: CNN Architecture — Data Flow Visualization', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# Model 3: The Organized Pattern Recognizer
# A convolutional network obtained by modifying the one based in the coursework description figure 1 & by adding a batch normalisation layer. 

In [ ]:
class MusicGenreCNN_BatchNorm(nn.Module):
    """ Modified CNN from Figure 1 with Batch Normalisation layers added.
    Figure 1 base (Model 2):
        Input → conv1→ReLU → conv2→ReLU → MaxPool
              → conv3→ReLU → conv4→ReLU → MaxPool
              → FC(out_features)→ReLU → FC(output)
Model 3 modification — ★BatchNorm inserted after each Conv2d and FC1:
        Input
        → Conv2d(in_channels=3,  out_channels=32,  kernel_size=3) + ★BatchNorm2d + ReLU
        → Conv2d(in_channels=32, out_channels=64,  kernel_size=3) + ★BatchNorm2d + ReLU
        → MaxPool2d(2x2)
        → Conv2d(in_channels=64,  out_channels=128, kernel_size=3) + ★BatchNorm2d + ReLU
        → Conv2d(in_channels=128, out_channels=128, kernel_size=3) + ★BatchNorm2d + ReLU
        → MaxPool2d(2x2)
        → Flatten
        → Linear(out_features=256) + ★BatchNorm1d + ReLU + Dropout
        → Linear(out_features=10)  [output]
    """

    def __init__(self, num_classes=10, dropout=0.4):
        super(MusicGenreCNN_BatchNorm, self).__init__()

        # ---- Convolutional feature extractor ----
        # (Same conv/pool structure as Figure 1, with ★BatchNorm2d added)

        # conv1: in_channels=3, out_channels=32, kernel_size=3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)                    # ★ Added

        # conv2: in_channels=32, out_channels=64, kernel_size=3
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)                    # ★ Added

        # MaxPool 1
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 180x180 → 90x90

        # conv3: in_channels=64, out_channels=128, kernel_size=3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)                   # ★ Added

        # conv4: in_channels=128, out_channels=128, kernel_size=3
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)                   # ★ Added

        # MaxPool 2
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # 90x90 → 45x45

        # ---- Classifier (fully connected layers) ----
        # After conv+pool: 128 channels × 45 × 45 = 259,200
        self.flatten = nn.Flatten()

        # FC1: out_features=256 (from Figure 1)
        self.fc1    = nn.Linear(128 * 45 * 45, 256)
        self.bn_fc1 = nn.BatchNorm1d(256)                  # ★ Added
        self.dropout = nn.Dropout(dropout)

        # FC2: output (num_classes)
        self.fc2    = nn.Linear(256, num_classes)

        # Activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # conv1 → BatchNorm → ReLU
        x = self.relu(self.bn1(self.conv1(x)))
        # conv2 → BatchNorm → ReLU
        x = self.relu(self.bn2(self.conv2(x)))
        # MaxPool 1
        x = self.pool1(x)

        # conv3 → BatchNorm → ReLU
        x = self.relu(self.bn3(self.conv3(x)))
        # conv4 → BatchNorm → ReLU
        x = self.relu(self.bn4(self.conv4(x)))
        # MaxPool 2
        x = self.pool2(x)

        # Flatten → FC1 → BatchNorm → ReLU → Dropout → FC2
        x = self.flatten(x)
        x = self.dropout(self.relu(self.bn_fc1(self.fc1(x))))
        x = self.fc2(x)
        return x


# --- Instantiate the BatchNorm CNN model ---
cnn_bn_model = MusicGenreCNN_BatchNorm(num_classes=10, dropout=0.4)

# Loss function and optimizer (Adam)
cnn_bn_criterion = nn.CrossEntropyLoss()
cnn_bn_optimizer = optim.Adam(cnn_bn_model.parameters(), lr=1e-3)

print(cnn_bn_model)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn_bn_model.parameters()):,}")

# --- Compare with base CNN (Model 2 — no BatchNorm) ---
base_params = sum(p.numel() for p in cnn_model.parameters())
bn_params   = sum(p.numel() for p in cnn_bn_model.parameters())
print(f"\n{'='*55}")
print(f"  Model 2 (Base CNN)       : {base_params:>12,} params")
print(f"  Model 3 (CNN + BatchNorm): {bn_params:>12,} params")
print(f"  Extra params from BN       : {bn_params - base_params:>12,} params")
print(f"{'='*55}")

# Verify output shape with a dummy input
# FIX: BatchNorm1d requires batch_size > 1 in training mode, so we
#      switch to eval() mode (uses running stats instead of batch stats)
cnn_bn_model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 180, 180)
    out = cnn_bn_model(dummy)
    print(f"\nInput shape:  {dummy.shape}")
    print(f"Output shape: {out.shape}")
cnn_bn_model.train()  # switch back to training mode for later use

# VISUALIZATION: Model 3 — BatchNorm Effect on Weight Distributions

In [ ]:
# ============================================================
# VISUALIZATION: Model 3 — BatchNorm Effect on Weight Distributions
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- 1. Weight distributions: Model 2 (no BN) vs Model 3 (with BN) ---
# Collect conv layer weights from both models
s2_weights = []
s3_weights = []

for name, param in cnn_model.named_parameters():
    if 'weight' in name and 'features' in name and param.dim() == 4:
        s2_weights.append(param.detach().cpu().numpy().flatten())

for name, param in cnn_bn_model.named_parameters():
    if 'weight' in name and 'conv' in name:
        s3_weights.append(param.detach().cpu().numpy().flatten())

conv_names = ['Conv1', 'Conv2', 'Conv3', 'Conv4']

# Top-left: Overlaid histograms for all conv weights
all_s2 = np.concatenate(s2_weights)
all_s3 = np.concatenate(s3_weights)
axes[0, 0].hist(all_s2, bins=80, alpha=0.6, color='#e74c3c', label='Model 2 (No BN)', density=True)
axes[0, 0].hist(all_s3, bins=80, alpha=0.6, color='#2ecc71', label='Model 3 (With BN)', density=True)
axes[0, 0].set_xlabel('Weight Value', fontsize=11)
axes[0, 0].set_ylabel('Density', fontsize=11)
axes[0, 0].set_title('All Conv Weights Distribution', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

# Top-right: Per-layer weight std comparison (grouped bar chart)
s2_stds = [w.std() for w in s2_weights]
s3_stds = [w.std() for w in s3_weights]
x = np.arange(len(conv_names))
width = 0.35
bars1 = axes[0, 1].bar(x - width/2, s2_stds, width, label='Model 2 (No BN)',
                         color='#e74c3c', edgecolor='white')
bars2 = axes[0, 1].bar(x + width/2, s3_stds, width, label='Model 3 (With BN)',
                         color='#2ecc71', edgecolor='white')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(conv_names, fontsize=11)
axes[0, 1].set_ylabel('Weight Std Dev', fontsize=11)
axes[0, 1].set_title('Per-Layer Weight Spread', fontsize=13, fontweight='bold')
axes[0, 1].legend(fontsize=10)

# --- 2. Architecture comparison: which layers have BatchNorm ---
layer_labels = ['Conv1', 'BN1', 'ReLU', 'Conv2', 'BN2', 'ReLU',
                'Pool', 'Conv3', 'BN3', 'ReLU', 'Conv4', 'BN4', 'ReLU',
                'Pool', 'FC1', 'BN_FC', 'ReLU', 'Drop', 'FC2']

# Color-code: orange=conv, green=BN, blue=activation, red=pool, purple=FC
color_map = {'Conv': '#e67e22', 'BN': '#2ecc71', 'ReLU': '#3498db',
             'Pool': '#e74c3c', 'FC': '#9b59b6', 'Drop': '#95a5a6'}

def get_color(label):
    for key in color_map:
        if key in label:
            return color_map[key]
    return '#bdc3c7'

colors_arch = [get_color(l) for l in layer_labels]
axes[1, 0].barh(range(len(layer_labels)), [1]*len(layer_labels),
                 color=colors_arch, edgecolor='white', linewidth=1.5, height=0.7)
axes[1, 0].set_yticks(range(len(layer_labels)))
axes[1, 0].set_yticklabels(layer_labels, fontsize=9)
axes[1, 0].set_xlim(0, 1.5)
axes[1, 0].set_xticks([])
axes[1, 0].set_title('Model 3: Full Layer Pipeline', fontsize=13, fontweight='bold')
axes[1, 0].invert_yaxis()
# Legend
legend_patches = [mpatches.Patch(color=c, label=l) for l, c in color_map.items()]
axes[1, 0].legend(handles=legend_patches, loc='lower right', fontsize=9, ncol=2)

# --- 3. BatchNorm learnable params (gamma & beta) initial values ---
bn_gammas = []
bn_betas = []
bn_names = []
for name, param in cnn_bn_model.named_parameters():
    if 'bn' in name and 'weight' in name:
        bn_gammas.append(param.detach().cpu().numpy())
        bn_names.append(name.replace('.weight', ''))
    elif 'bn' in name and 'bias' in name:
        bn_betas.append(param.detach().cpu().numpy())

x_bn = np.arange(len(bn_names))
axes[1, 1].boxplot(bn_gammas, positions=x_bn - 0.2, widths=0.35,
                    patch_artist=True, boxprops=dict(facecolor='#2ecc71', alpha=0.7),
                    medianprops=dict(color='black'))
axes[1, 1].boxplot(bn_betas, positions=x_bn + 0.2, widths=0.35,
                    patch_artist=True, boxprops=dict(facecolor='#3498db', alpha=0.7),
                    medianprops=dict(color='black'))
axes[1, 1].set_xticks(x_bn)
axes[1, 1].set_xticklabels([n.replace('_', '\n') for n in bn_names], fontsize=9)
axes[1, 1].set_ylabel('Parameter Value', fontsize=11)
axes[1, 1].set_title('BatchNorm Params: Gamma (green) & Beta (blue)', fontsize=13, fontweight='bold')
axes[1, 1].axhline(y=1.0, color='#2ecc71', linestyle='--', alpha=0.5, label='gamma init=1')
axes[1, 1].axhline(y=0.0, color='#3498db', linestyle='--', alpha=0.5, label='beta init=0')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Model 3: Impact of Batch Normalisation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Model 4: The Same Organized Model, Different Study Method
# The same architecture in the above 3, with the RMSProp optimiser.

In [ ]:
class MusicGenreCNN_BatchNorm_RMSProp(nn.Module):
    """
    Same CNN + BatchNorm architecture as Model 3, but trained with RMSProp optimiser.

    Architecture (identical to MusicGenreCNN_BatchNorm):
        Input (3x180x180)
        → Conv1 (3→32, 3x3)  + BatchNorm2d(32)  + ReLU
        → Conv2 (32→64, 3x3) + BatchNorm2d(64)  + ReLU
        → MaxPool (2x2)
        → Conv3 (64→128, 3x3)  + BatchNorm2d(128) + ReLU
        → Conv4 (128→128, 3x3) + BatchNorm2d(128) + ReLU
        → MaxPool (2x2)
        → Flatten
        → FC1 (→256) + BatchNorm1d(256) + ReLU + Dropout
        → FC2 (→10)  [Output]

    Key difference: Uses ★RMSProp optimiser instead of Adam.
    """

    def __init__(self, num_classes=10, dropout=0.4):
        super(MusicGenreCNN_BatchNorm_RMSProp, self).__init__()

        # --- Convolutional feature extractor (with BatchNorm) ---
        self.features = nn.Sequential(
            # Conv1 + BatchNorm + ReLU
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            # Conv2 + BatchNorm + ReLU
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            # MaxPool 1
            nn.MaxPool2d(kernel_size=2, stride=2),        # 180x180 → 90x90

            # Conv3 + BatchNorm + ReLU
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # Conv4 + BatchNorm + ReLU
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # MaxPool 2
            nn.MaxPool2d(kernel_size=2, stride=2),        # 90x90 → 45x45
        )

        # --- Classifier (with BatchNorm in FC layer) ---
        # After features: 128 channels x 45 x 45 = 259,200
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 45 * 45, 256),                # FC1
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),                   # FC2 (Output)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# --- Instantiate the model ---
cnn_bn_rms_model = MusicGenreCNN_BatchNorm_RMSProp(num_classes=10, dropout=0.4)

# ★ Loss function and RMSProp optimiser (instead of Adam)
cnn_bn_rms_criterion = nn.CrossEntropyLoss()
cnn_bn_rms_optimizer = optim.RMSprop(
    cnn_bn_rms_model.parameters(),
    lr=1e-3,          # learning rate
    alpha=0.99,       # smoothing constant (decay factor for running average of squared gradients)
    momentum=0.0,     # momentum factor
    weight_decay=1e-4 # L2 regularisation
)

print(cnn_bn_rms_model)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn_bn_rms_model.parameters()):,}")

# --- Compare optimisers: Model 3 (Adam) vs Model 4 (RMSProp) ---
print(f"\n{'='*55}")
print(f"  Model 3 optimiser: Adam      (lr=1e-3)")
print(f"  Model 4 optimiser: ★RMSProp  (lr=1e-3, alpha=0.99, weight_decay=1e-4)")
print(f"{'='*55}")

# Verify output shape with a dummy input
# FIX: BatchNorm1d requires batch_size > 1 in training mode, so we
#      switch to eval() mode (uses running stats instead of batch stats)
cnn_bn_rms_model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 180, 180)
    out = cnn_bn_rms_model(dummy)
    print(f"\nInput shape:  {dummy.shape}")
    print(f"Output shape: {out.shape}")
cnn_bn_rms_model.train()  # switch back to training mode for later use

# VISUALIZATION: Model 4 — All Models Comparison (Radar + Summary Table)

In [ ]:
# ============================================================
# VISUALIZATION: Model 4 — All Models Comparison (Radar + Summary Table)
# ============================================================

fig = plt.figure(figsize=(18, 8))

# --- 1. Radar Chart: Multi-dimensional comparison of all 4 models ---
ax_radar = fig.add_subplot(121, polar=True)

categories = ['Parameters\n(normalized)', 'Depth\n(layers)', 'Regularisation\n(BN+Dropout)',
              'Optimiser\nSophistication', 'Architecture\nComplexity']
N = len(categories)

# Normalize scores (0-1 scale) for fair comparison
model_scores = {
    'Model 1\n(FC)':           [0.01, 0.3, 0.6, 0.7, 0.2],
    'Model 2\n(CNN)':          [1.0,  0.7, 0.3, 0.7, 0.7],
    'Model 3\n(CNN+BN)':       [1.0,  0.8, 0.9, 0.7, 0.85],
    'Model 4\n(CNN+BN+RMSProp)':[1.0, 0.8, 0.9, 0.5, 0.85],
}

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors_radar = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
for idx, (name, values) in enumerate(model_scores.items()):
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=name, color=colors_radar[idx])
    ax_radar.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, fontsize=9)
ax_radar.set_ylim(0, 1.1)
ax_radar.set_title('Multi-Dimensional Model Comparison', fontsize=13, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

# --- 2. Grouped Bar Chart: Key metrics side by side ---
ax_bar = fig.add_subplot(122)

models = ['Model 1\n(FC)', 'Model 2\n(CNN)', 'Model 3\n(CNN+BN)', 'Model 4\n(CNN+BN\n+RMSProp)']
param_counts_all = [
    sum(p.numel() for p in model.parameters()),
    sum(p.numel() for p in cnn_model.parameters()),
    sum(p.numel() for p in cnn_bn_model.parameters()),
    sum(p.numel() for p in cnn_bn_rms_model.parameters())
]

log_params = [np.log10(p) for p in param_counts_all]
bars = ax_bar.bar(models, log_params, color=colors_radar, edgecolor='white', linewidth=2, width=0.6)
for bar, count in zip(bars, param_counts_all):
    ax_bar.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                f'{count:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_bar.set_ylabel('log10(Parameters)', fontsize=12)
ax_bar.set_title('Total Parameters Per Model (log scale)', fontsize=13, fontweight='bold')
ax_bar.set_ylim(0, max(log_params) + 0.8)

plt.suptitle('All 4 Models — Comparison Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 3. Summary Table ---
print("\n" + "="*85)
print(f"{'Model COMPARISON TABLE':^85}")
print("="*85)
print(f"{'Property':<25} {'Model 1':<15} {'Model 2':<15} {'Model 3':<15} {'Model 4':<15}")
print("-"*85)
print(f"{'Type':<25} {'FC':<15} {'CNN':<15} {'CNN+BN':<15} {'CNN+BN':<15}")
print(f"{'Parameters':<25} {param_counts_all[0]:<15,} {param_counts_all[1]:<15,} {param_counts_all[2]:<15,} {param_counts_all[3]:<15,}")
print(f"{'Conv Layers':<25} {'0':<15} {'4':<15} {'4':<15} {'4':<15}")
print(f"{'BatchNorm Layers':<25} {'2 (1d)':<15} {'0':<15} {'5 (4x2d+1d)':<15} {'5 (4x2d+1d)':<15}")
print(f"{'Dropout':<25} {'0.3':<15} {'0.4':<15} {'0.4':<15} {'0.4':<15}")
print(f"{'Optimiser':<25} {'Adam':<15} {'Adam':<15} {'Adam':<15} {'RMSProp':<15}")
print(f"{'Learning Rate':<25} {'1e-3':<15} {'1e-3':<15} {'1e-3':<15} {'1e-3':<15}")
print("="*85)

In [ ]:
# ============================================================
# VISUALIZATION: Compare All Students After 100 Epochs
# ============================================================

histories_100 = {
    'Student 2 (CNN)':            history_s2_100,
    'Student 3 (CNN+BN)':        history_s3_100,
    'Student 4 (CNN+BN+RMSProp)': history_s4_100,
}

plot_compare_histories(histories_100)

# Model 5: The Sequence Learner — RNN with LSTMs
# An LSTM-based recurrent network that treats the Mel spectrogram as a time series.
# Each of the 1292 time frames is a step, with 128 Mel-band values as the feature vector.
# A 2-layer bidirectional LSTM captures temporal patterns, and the final hidden states
# are fed through a fully connected classifier to predict the genre.

In [ ]:
class MusicGenreLSTM(nn.Module):
    """
    LSTM-based RNN for music genre classification from Mel spectrograms.

    The Mel spectrogram (1, 128, 1292) is reshaped so that each of the 1292
    time frames becomes a time step, and the 128 Mel bands are the input features
    at each step.

    Architecture:
        Input Mel spectrogram (1, 128, 1292)
        → Reshape to (1292, 128)  [sequence_length, input_size]
        → LSTM (128→256, 2 layers, bidirectional, dropout=0.3)
        → Concatenate final forward + backward hidden states (512)
        → FC1 (512→128) + BatchNorm1d + ReLU + Dropout(0.3)
        → FC2 (128→10)  [Output]
    """

    def __init__(self, input_size=128, hidden_size=256, num_layers=2,
                 num_classes=10, dropout=0.3, bidirectional=True):
        super(MusicGenreLSTM, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        # --- LSTM layers ---
        self.lstm = nn.LSTM(
            input_size=input_size,       # 128 Mel bands per time step
            hidden_size=hidden_size,      # 256 hidden units
            num_layers=num_layers,         # 2 stacked LSTM layers
            batch_first=True,              # input shape: (batch, seq_len, features)
            dropout=dropout if num_layers > 1 else 0,  # dropout between LSTM layers
            bidirectional=bidirectional    # process sequence forward + backward
        )

        # --- Classifier head ---
        fc_input = hidden_size * self.num_directions  # 512 if bidirectional
        self.classifier = nn.Sequential(
            nn.Linear(fc_input, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x shape: (batch, 1, 128, 1292) — Mel spectrogram with channel dim
        # Remove channel dim and transpose to (batch, time_steps, mel_bands)
        x = x.squeeze(1)              # (batch, 128, 1292)
        x = x.permute(0, 2, 1)        # (batch, 1292, 128)

        # LSTM forward pass
        # lstm_out: (batch, 1292, hidden_size * num_directions)
        # h_n:      (num_layers * num_directions, batch, hidden_size)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Extract final hidden states from the last LSTM layer
        # Forward direction: h_n[-2]  |  Backward direction: h_n[-1]
        if self.bidirectional:
            h_forward  = h_n[-2]       # (batch, hidden_size)
            h_backward = h_n[-1]       # (batch, hidden_size)
            hidden = torch.cat([h_forward, h_backward], dim=1)  # (batch, 512)
        else:
            hidden = h_n[-1]           # (batch, hidden_size)

        # Classify
        out = self.classifier(hidden)
        return out


# --- Instantiate Model 5 ---
lstm_model = MusicGenreLSTM(
    input_size=N_MELS,         # 128 Mel bands
    hidden_size=256,
    num_layers=2,
    num_classes=10,
    dropout=0.3,
    bidirectional=True
)

lstm_criterion = nn.CrossEntropyLoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3)

print(lstm_model)
print(f"\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

# --- Verify output shape ---
lstm_model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 1, N_MELS, EXPECTED_FRAMES)  # (batch=2, 1, 128, 1292)
    out = lstm_model(dummy)
    print(f"\nInput shape:  {dummy.shape}  (batch, channel, mel_bands, time_frames)")
    print(f"Output shape: {out.shape}   (batch, num_classes)")
lstm_model.train()

# --- Architecture summary ---
print(f"\n{'='*60}")
print(f"  Model 5: LSTM (Bidirectional, 2 layers)")
print(f"  Input:     Mel spectrogram (1, {N_MELS}, {EXPECTED_FRAMES})")
print(f"  Sequence:  {EXPECTED_FRAMES} time steps x {N_MELS} features")
print(f"  LSTM:      {256} hidden units x 2 layers x 2 directions")
print(f"  Classifier: 512 → 128 → 10")
print(f"  Optimiser: Adam (lr=1e-3)")
print(f"{'='*60}")


# VISUALIZATION: Model 5 — LSTM Sequence Processing Pipeline

In [ ]:
# ============================================================
# VISUALIZATION: Model 5 — LSTM Sequence Processing Pipeline
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- 1. LSTM Unrolled Architecture Diagram ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('LSTM Unrolled Architecture', fontsize=13, fontweight='bold')

# Draw time steps
time_labels = ['t=1', 't=2', '...', 't=1291', 't=1292']
x_positions = [1, 2.5, 4.5, 6.5, 8]
colors_lstm = ['#3498db', '#3498db', '#95a5a6', '#3498db', '#e74c3c']

for i, (xp, label, color) in enumerate(zip(x_positions, time_labels, colors_lstm)):
    # LSTM cell box
    rect = plt.Rectangle((xp - 0.5, 4), 1.0, 1.5, facecolor=color,
                          edgecolor='black', linewidth=1.5, alpha=0.7)
    ax.add_patch(rect)
    ax.text(xp, 4.75, f'LSTM\n{label}', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

    # Input arrow from below
    ax.annotate('', xy=(xp, 4), xytext=(xp, 2.8),
               arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))
    ax.text(xp, 2.4, f'128\nMel', ha='center', va='center', fontsize=7, color='#2c3e50')

    # Horizontal arrows between cells
    if i < len(x_positions) - 1:
        next_xp = x_positions[i + 1]
        ax.annotate('', xy=(next_xp - 0.5, 4.75), xytext=(xp + 0.5, 4.75),
                   arrowprops=dict(arrowstyle='->', color='#e67e22', lw=2))

# Output arrow from last cell
ax.annotate('', xy=(8, 7), xytext=(8, 5.5),
           arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
ax.text(8, 7.5, 'h_final\n(512)', ha='center', va='center',
        fontsize=9, fontweight='bold', color='#e74c3c',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fadbd8', edgecolor='#e74c3c'))

# Bidirectional backward arrow
ax.annotate('', xy=(1, 3.6), xytext=(8, 3.6),
           arrowprops=dict(arrowstyle='->', color='#9b59b6', lw=1.5, linestyle='--'))
ax.text(4.5, 3.2, '← Backward direction', ha='center', fontsize=8,
        color='#9b59b6', fontstyle='italic')
ax.text(4.5, 6.2, 'Forward direction →', ha='center', fontsize=8,
        color='#e67e22', fontstyle='italic')

# --- 2. Parameter Distribution (Pie Chart) ---
ax = axes[1]
param_groups = {}
for name, param in lstm_model.named_parameters():
    if 'lstm' in name:
        group = 'LSTM Layers'
    elif 'classifier.0' in name:
        group = 'FC1 (512→128)'
    elif 'classifier.1' in name:
        group = 'BatchNorm1d'
    elif 'classifier.4' in name:
        group = 'FC2 (128→10)'
    else:
        group = 'Other'
    param_groups[group] = param_groups.get(group, 0) + param.numel()

labels_pie = list(param_groups.keys())
sizes_pie = list(param_groups.values())
colors_pie = ['#3498db', '#2ecc71', '#f1c40f', '#e74c3c', '#95a5a6']
explode = [0.05] * len(labels_pie)

wedges, texts, autotexts = ax.pie(
    sizes_pie, labels=labels_pie, autopct='%1.1f%%',
    colors=colors_pie[:len(labels_pie)], explode=explode[:len(labels_pie)],
    shadow=True, startangle=140, textprops={'fontsize': 9}
)
ax.set_title('Parameter Distribution — Model 5 (LSTM)', fontsize=13, fontweight='bold')

# --- 3. Model Comparison: All 5 Models ---
ax = axes[2]
all_models_info = [
    ('Model 1\n(FC)', sum(p.numel() for p in model.parameters()), '#3498db'),
    ('Model 2\n(CNN)', sum(p.numel() for p in cnn_model.parameters()), '#e67e22'),
    ('Model 3\n(CNN+BN)', sum(p.numel() for p in cnn_bn_model.parameters()), '#2ecc71'),
    ('Model 4\n(CNN+BN\n+RMSProp)', sum(p.numel() for p in cnn_bn_rms_model.parameters()), '#e74c3c'),
    ('Model 5\n(LSTM)', sum(p.numel() for p in lstm_model.parameters()), '#9b59b6'),
]

names = [m[0] for m in all_models_info]
params = [m[1] for m in all_models_info]
colors = [m[2] for m in all_models_info]

bars = ax.bar(names, [np.log10(p) for p in params], color=colors,
              edgecolor='white', linewidth=2, width=0.6)
for bar, count in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{count:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('log10(Parameters)', fontsize=11)
ax.set_title('All 5 Models — Parameter Count', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(np.log10(p) for p in params) + 0.8)

plt.suptitle('Model 5 — LSTM Sequence Learner', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# Model 6 - The Well-Practiced Timeline Listener
# The same RNN but with extra fake music samples created by GANs to help it learn better

In [ ]:
# ============================================================
# MODEL 6 — LSTM + GAN Data Augmentation
# ============================================================
# Step 1: Train a conditional GAN to generate synthetic Mel spectrograms
# Step 2: Generate fake samples equal to the number of real training samples
# Step 3: Combine real + fake into an augmented dataset
# Step 4: Instantiate the same LSTM classifier (Model 5 architecture)
# ============================================================

# ------------------------------------------------------------------
# 1. GAN ARCHITECTURE — Conditional Generator & Discriminator
#    Both are conditioned on the genre label (10 classes) so the GAN
#    learns to produce spectrograms for each genre independently.
# ------------------------------------------------------------------

class MelGenerator(nn.Module):
    """
    Conditional Generator: noise vector z (100) + label embedding (50)
    → FC → Reshape → 4x ConvTranspose1d → Mel spectrogram (128 x 1292)
    
    Generates one Mel spectrogram at a time, treating it as a 1-D signal
    with 128 channels (one per Mel band) across 1292 time frames.
    """

    def __init__(self, noise_dim=100, label_dim=50, num_classes=10,
                 n_mels=128, n_frames=1292):
        super(MelGenerator, self).__init__()

        self.n_mels = n_mels
        self.n_frames = n_frames
        self.init_frames = n_frames // 16  # 80 (will upsample 16x back to ~1280)

        # Label embedding: integer label → dense vector
        self.label_emb = nn.Embedding(num_classes, label_dim)

        # FC: project (noise + label) into a volume we can reshape
        self.fc = nn.Sequential(
            nn.Linear(noise_dim + label_dim, 512 * self.init_frames),
            nn.BatchNorm1d(512 * self.init_frames),
            nn.ReLU(True),
        )

        # Upsampling convolutions (1-D along time axis)
        self.conv_blocks = nn.Sequential(
            # 512 x 80 → 256 x 160
            nn.ConvTranspose1d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(True),

            # 256 x 160 → 128 x 320
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(True),

            # 128 x 320 → 128 x 640
            nn.ConvTranspose1d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(True),

            # 128 x 640 → 128 x 1280
            nn.ConvTranspose1d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(True),

            # Final 1x1 conv to refine + Tanh output
            nn.Conv1d(128, n_mels, kernel_size=1),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        # z: (batch, noise_dim), labels: (batch,)
        label_vec = self.label_emb(labels)           # (batch, label_dim)
        x = torch.cat([z, label_vec], dim=1)         # (batch, noise_dim + label_dim)
        x = self.fc(x)                                # (batch, 512 * init_frames)
        x = x.view(-1, 512, self.init_frames)         # (batch, 512, 80)
        x = self.conv_blocks(x)                        # (batch, 128, 1280)
        # Pad or trim to exact n_frames
        if x.size(2) < self.n_frames:
            x = nn.functional.pad(x, (0, self.n_frames - x.size(2)))
        else:
            x = x[:, :, :self.n_frames]
        return x  # (batch, 128, 1292)


class MelDiscriminator(nn.Module):
    """
    Conditional Discriminator: Mel spectrogram (128 x 1292) + label embedding
    → 4x Conv1d → FC → real/fake score
    """

    def __init__(self, label_dim=50, num_classes=10, n_mels=128, n_frames=1292):
        super(MelDiscriminator, self).__init__()

        self.n_frames = n_frames
        self.label_emb = nn.Embedding(num_classes, label_dim)

        # Input channels = n_mels + label_dim (label broadcast across time)
        in_ch = n_mels + label_dim

        self.conv_blocks = nn.Sequential(
            # (128+50) x 1292 → 128 x 646
            nn.Conv1d(in_ch, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            # 128 x 646 → 256 x 323
            nn.Conv1d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # 256 x 323 → 512 x 161
            nn.Conv1d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),

            # 512 x 161 → 512 x 80
            nn.Conv1d(512, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Adaptive pooling to collapse time axis, then linear → scalar
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),   # (batch, 512, 1)
            nn.Flatten(),              # (batch, 512)
            nn.Linear(512, 1),         # (batch, 1)
        )

    def forward(self, mel, labels):
        # mel: (batch, 128, 1292), labels: (batch,)
        label_vec = self.label_emb(labels)                       # (batch, 50)
        label_vec = label_vec.unsqueeze(2).expand(-1, -1, self.n_frames)  # (batch, 50, 1292)
        x = torch.cat([mel, label_vec], dim=1)                   # (batch, 178, 1292)
        x = self.conv_blocks(x)
        return self.head(x)  # (batch, 1)


# ------------------------------------------------------------------
# 2. TRAIN THE GAN
# ------------------------------------------------------------------

NOISE_DIM = 100
GAN_EPOCHS = 30
GAN_LR = 2e-4
GAN_BATCH = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

generator = MelGenerator(noise_dim=NOISE_DIM, n_mels=N_MELS, n_frames=EXPECTED_FRAMES).to(device)
discriminator = MelDiscriminator(n_mels=N_MELS, n_frames=EXPECTED_FRAMES).to(device)

opt_G = optim.Adam(generator.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
opt_D = optim.Adam(discriminator.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
criterion_gan = nn.BCEWithLogitsLoss()

print(f"Generator parameters:     {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")
print(f"Training GAN for {GAN_EPOCHS} epochs on device: {device}\n")

# Build a DataLoader from the TRAINING split only (no channel dim for GAN)
# audio_train_loader provides (batch, 1, 128, 1292) tensors
# We need (batch, 128, 1292) for the GAN
gan_train_data = []
gan_train_labels = []
for batch_X, batch_y in audio_train_loader:
    gan_train_data.append(batch_X.squeeze(1))   # remove channel dim
    gan_train_labels.append(batch_y)
gan_train_data = torch.cat(gan_train_data, dim=0)
gan_train_labels = torch.cat(gan_train_labels, dim=0)

# Normalise real Mel spectrograms to [-1, 1] for Tanh generator output
mel_min = gan_train_data.min()
mel_max = gan_train_data.max()
gan_train_data_norm = 2.0 * (gan_train_data - mel_min) / (mel_max - mel_min + 1e-8) - 1.0

gan_dataset = TensorDataset(gan_train_data_norm, gan_train_labels)
gan_loader = DataLoader(gan_dataset, batch_size=GAN_BATCH, shuffle=True, drop_last=True)

num_real_train = len(gan_train_data)
print(f"Real training samples: {num_real_train}")
print(f"Will generate {num_real_train} synthetic samples after training\n")

# --- GAN Training Loop ---
g_losses = []
d_losses = []

for epoch in range(1, GAN_EPOCHS + 1):
    g_loss_epoch = 0.0
    d_loss_epoch = 0.0
    n_batches = 0

    for real_mel, real_labels in gan_loader:
        batch_size = real_mel.size(0)
        real_mel = real_mel.to(device)
        real_labels = real_labels.to(device)

        real_target = torch.ones(batch_size, 1, device=device)
        fake_target = torch.zeros(batch_size, 1, device=device)

        # --- Train Discriminator ---
        z = torch.randn(batch_size, NOISE_DIM, device=device)
        fake_mel = generator(z, real_labels)

        d_real = discriminator(real_mel, real_labels)
        d_fake = discriminator(fake_mel.detach(), real_labels)
        loss_D = criterion_gan(d_real, real_target) + criterion_gan(d_fake, fake_target)

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # --- Train Generator ---
        z = torch.randn(batch_size, NOISE_DIM, device=device)
        fake_mel = generator(z, real_labels)
        d_fake = discriminator(fake_mel, real_labels)
        loss_G = criterion_gan(d_fake, real_target)  # fool discriminator

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

        g_loss_epoch += loss_G.item()
        d_loss_epoch += loss_D.item()
        n_batches += 1

    g_losses.append(g_loss_epoch / n_batches)
    d_losses.append(d_loss_epoch / n_batches)

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{GAN_EPOCHS}  |  D loss: {d_losses[-1]:.4f}  |  G loss: {g_losses[-1]:.4f}")

print("\nGAN training complete.")

# ------------------------------------------------------------------
# 3. GENERATE SYNTHETIC SAMPLES (same count as real training set)
# ------------------------------------------------------------------

generator.eval()

# Generate label-balanced fakes: same class distribution as real training data
unique_labels, label_counts = torch.unique(gan_train_labels, return_counts=True)
print(f"\nGenerating {num_real_train} synthetic Mel spectrograms...")

fake_mels = []
fake_labels_list = []

with torch.no_grad():
    for lbl, cnt in zip(unique_labels, label_counts):
        n = cnt.item()
        # Generate in sub-batches to avoid OOM
        generated = 0
        while generated < n:
            sub_batch = min(16, n - generated)
            z = torch.randn(sub_batch, NOISE_DIM, device=device)
            lbl_tensor = torch.full((sub_batch,), lbl.item(), dtype=torch.long, device=device)
            fake = generator(z, lbl_tensor).cpu()
            fake_mels.append(fake)
            fake_labels_list.append(lbl_tensor.cpu())
            generated += sub_batch

fake_mels = torch.cat(fake_mels, dim=0)       # (N_fake, 128, 1292) in [-1, 1]
fake_labels_all = torch.cat(fake_labels_list, dim=0)

# Denormalise back to dB scale
fake_mels = (fake_mels + 1.0) / 2.0 * (mel_max - mel_min + 1e-8) + mel_min

print(f"Generated tensor shape: {fake_mels.shape}")
print(f"Generated labels shape: {fake_labels_all.shape}")

# ------------------------------------------------------------------
# 4. BUILD AUGMENTED DATASET (real + fake) AND SPLIT
# ------------------------------------------------------------------

# Combine real training data + generated data
augmented_data = torch.cat([gan_train_data, fake_mels], dim=0)          # (2N, 128, 1292)
augmented_labels = torch.cat([gan_train_labels, fake_labels_all], dim=0)  # (2N,)

# Add channel dimension for the LSTM model input
augmented_data = augmented_data.unsqueeze(1)  # (2N, 1, 128, 1292)

print(f"\nAugmented training set: {augmented_data.shape[0]} samples "
      f"({num_real_train} real + {fake_mels.shape[0]} synthetic)")

augmented_train_dataset = TensorDataset(augmented_data, augmented_labels)
augmented_train_loader = DataLoader(
    augmented_train_dataset, batch_size=32, shuffle=True, drop_last=True
)

# Use the ORIGINAL validation and test loaders (no augmentation on eval sets)
# These are audio_val_loader and audio_test_loader from the audio loading cell

print(f"Augmented train loader: {len(augmented_train_loader)} batches")
print(f"Validation loader:     {len(audio_val_loader)} batches (original, no augmentation)")
print(f"Test loader:           {len(audio_test_loader)} batches (original, no augmentation)")

# ------------------------------------------------------------------
# 5. INSTANTIATE MODEL 6 — Same LSTM architecture as Model 5
# ------------------------------------------------------------------

lstm_gan_model = MusicGenreLSTM(
    input_size=N_MELS,         # 128 Mel bands
    hidden_size=256,
    num_layers=2,
    num_classes=10,
    dropout=0.3,
    bidirectional=True
)

lstm_gan_criterion = nn.CrossEntropyLoss()
lstm_gan_optimizer = optim.Adam(lstm_gan_model.parameters(), lr=1e-3)

print(f"\nModel 6 (LSTM + GAN augmentation)")
print(f"  Same architecture as Model 5: {sum(p.numel() for p in lstm_gan_model.parameters()):,} parameters")
print(f"  Training data: {len(augmented_train_dataset)} samples (2x real via GAN augmentation)")

# --- Verify output shape ---
lstm_gan_model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 1, N_MELS, EXPECTED_FRAMES)
    out = lstm_gan_model(dummy)
    print(f"  Input:  {dummy.shape}  →  Output: {out.shape}")
lstm_gan_model.train()

# --- Summary ---
print(f"\n{'='*65}")
print(f"  Model 6: LSTM + GAN Data Augmentation")
print(f"  Classifier:     Bidirectional LSTM (same as Model 5)")
print(f"  GAN type:       Conditional GAN (label-aware generation)")
print(f"  GAN epochs:     {GAN_EPOCHS}")
print(f"  Real samples:   {num_real_train}")
print(f"  Fake samples:   {fake_mels.shape[0]}")
print(f"  Total training: {len(augmented_train_dataset)}")
print(f"  Optimiser:      Adam (lr=1e-3)")
print(f"{'='*65}")

# VISUALIZATION: Model 6 — GAN Training & Augmented Data Overview

In [ ]:
# ============================================================
# VISUALIZATION: Model 6 — GAN Training & Augmented Data Overview
# ============================================================

fig = plt.figure(figsize=(20, 12))

# --- 1. GAN Training Loss Curves ---
ax1 = fig.add_subplot(2, 3, 1)
epochs_range = range(1, len(g_losses) + 1)
ax1.plot(epochs_range, d_losses, 'b-o', label='Discriminator', markersize=4, linewidth=2)
ax1.plot(epochs_range, g_losses, 'r-s', label='Generator', markersize=4, linewidth=2)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('GAN Training Losses', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- 2. Real vs Fake Mel Spectrogram Comparison ---
ax2 = fig.add_subplot(2, 3, 2)
# Show a random real sample
sample_idx = np.random.randint(len(gan_train_data))
ax2.imshow(gan_train_data[sample_idx].numpy(), aspect='auto', origin='lower', cmap='magma')
genre_name = audio_label_encoder.classes_[gan_train_labels[sample_idx].item()]
ax2.set_title(f'Real Mel Spectrogram ({genre_name})', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time Frame', fontsize=10)
ax2.set_ylabel('Mel Band', fontsize=10)

ax3 = fig.add_subplot(2, 3, 3)
# Show a generated sample of the same genre
fake_idx = (fake_labels_all == gan_train_labels[sample_idx]).nonzero(as_tuple=True)[0][0]
ax3.imshow(fake_mels[fake_idx].numpy(), aspect='auto', origin='lower', cmap='magma')
ax3.set_title(f'Generated Mel Spectrogram ({genre_name})', fontsize=12, fontweight='bold')
ax3.set_xlabel('Time Frame', fontsize=10)
ax3.set_ylabel('Mel Band', fontsize=10)

# --- 3. Augmented Dataset Composition (Stacked Bar) ---
ax4 = fig.add_subplot(2, 3, 4)
genre_names = audio_label_encoder.classes_
real_counts = []
fake_counts = []
for i in range(len(genre_names)):
    real_counts.append((gan_train_labels == i).sum().item())
    fake_counts.append((fake_labels_all == i).sum().item())

x = np.arange(len(genre_names))
width = 0.6
bars_real = ax4.bar(x, real_counts, width, label='Real', color='#3498db', alpha=0.8)
bars_fake = ax4.bar(x, fake_counts, width, bottom=real_counts, label='Generated (GAN)',
                    color='#e74c3c', alpha=0.8)
ax4.set_xlabel('Genre', fontsize=11)
ax4.set_ylabel('Samples', fontsize=11)
ax4.set_title('Augmented Dataset Composition', fontsize=13, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(genre_names, rotation=45, ha='right', fontsize=9)
ax4.legend(fontsize=10)

# --- 4. GAN Architecture Flow Diagram ---
ax5 = fig.add_subplot(2, 3, 5)
ax5.set_xlim(0, 10)
ax5.set_ylim(0, 10)
ax5.axis('off')
ax5.set_title('Conditional GAN Architecture', fontsize=13, fontweight='bold')

# Noise box
rect = plt.Rectangle((0.3, 7), 2, 1.2, facecolor='#f39c12', edgecolor='black', lw=1.5, alpha=0.7)
ax5.add_patch(rect)
ax5.text(1.3, 7.6, 'Noise z\n(100-d)', ha='center', va='center', fontsize=8, fontweight='bold')

# Label box
rect = plt.Rectangle((0.3, 5.2), 2, 1.2, facecolor='#9b59b6', edgecolor='black', lw=1.5, alpha=0.7)
ax5.add_patch(rect)
ax5.text(1.3, 5.8, 'Genre\nLabel', ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# Generator
rect = plt.Rectangle((3.5, 6), 2.5, 1.5, facecolor='#2ecc71', edgecolor='black', lw=2, alpha=0.8)
ax5.add_patch(rect)
ax5.text(4.75, 6.75, 'Generator\n(ConvT1d)', ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows to generator
ax5.annotate('', xy=(3.5, 7.2), xytext=(2.3, 7.6), arrowprops=dict(arrowstyle='->', lw=1.5))
ax5.annotate('', xy=(3.5, 6.5), xytext=(2.3, 5.8), arrowprops=dict(arrowstyle='->', lw=1.5))

# Fake mel
rect = plt.Rectangle((7, 6), 2.5, 1.5, facecolor='#e74c3c', edgecolor='black', lw=1.5, alpha=0.7)
ax5.add_patch(rect)
ax5.text(8.25, 6.75, 'Fake Mel\n(128x1292)', ha='center', va='center', fontsize=8, fontweight='bold', color='white')
ax5.annotate('', xy=(7, 6.75), xytext=(6, 6.75), arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71'))

# Discriminator
rect = plt.Rectangle((3.5, 2), 2.5, 1.5, facecolor='#3498db', edgecolor='black', lw=2, alpha=0.8)
ax5.add_patch(rect)
ax5.text(4.75, 2.75, 'Discriminator\n(Conv1d)', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# Real mel
rect = plt.Rectangle((0.3, 2), 2, 1.2, facecolor='#2ecc71', edgecolor='black', lw=1.5, alpha=0.5)
ax5.add_patch(rect)
ax5.text(1.3, 2.6, 'Real Mel', ha='center', va='center', fontsize=8, fontweight='bold')
ax5.annotate('', xy=(3.5, 2.75), xytext=(2.3, 2.6), arrowprops=dict(arrowstyle='->', lw=1.5))

# Fake to discriminator
ax5.annotate('', xy=(5.5, 3.5), xytext=(8.25, 6), arrowprops=dict(arrowstyle='->', lw=1.5, linestyle='--', color='#e74c3c'))

# Output
ax5.text(7.5, 2.75, 'Real / Fake?', ha='center', va='center', fontsize=9, fontweight='bold', color='#e74c3c')
ax5.annotate('', xy=(7, 2.75), xytext=(6, 2.75), arrowprops=dict(arrowstyle='->', lw=2, color='#3498db'))

# --- 5. Model 5 vs Model 6 Comparison Table ---
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
ax6.set_title('Model 5 vs Model 6 Comparison', fontsize=13, fontweight='bold')

table_data = [
    ['Property', 'Model 5', 'Model 6'],
    ['Classifier', 'Bi-LSTM (2L)', 'Bi-LSTM (2L)'],
    ['LSTM Hidden', '256 x 2 dir', '256 x 2 dir'],
    ['Classifier Params', f'{sum(p.numel() for p in lstm_model.parameters()):,}',
                          f'{sum(p.numel() for p in lstm_gan_model.parameters()):,}'],
    ['Training Data', f'{num_real_train} (real)', f'{len(augmented_train_dataset)} (real+GAN)'],
    ['GAN Augmentation', 'No', 'Yes'],
    ['GAN Epochs', '-', f'{GAN_EPOCHS}'],
    ['Optimiser', 'Adam', 'Adam'],
]

table = ax6.table(cellText=table_data, cellLoc='center', loc='center',
                  colWidths=[0.35, 0.3, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

# Style header row
for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
# Alternate row colours
for i in range(1, len(table_data)):
    color = '#ecf0f1' if i % 2 == 0 else 'white'
    for j in range(3):
        table[i, j].set_facecolor(color)

plt.suptitle('Model 6 — LSTM + GAN Data Augmentation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## Final Comparison: 50 Epochs vs 100 Epochs

In [ ]:
# ============================================================
# FINAL VISUALIZATION: 50 Epochs vs 100 Epochs — Side-by-Side
# ============================================================

student_names = ['Student 2\n(CNN)', 'Student 3\n(CNN+BN)', 'Student 4\n(CNN+BN\n+RMSProp)']
colors_final = ['#e67e22', '#2ecc71', '#e74c3c']

# Collect best metrics
best_acc_50  = [max(history_s2_50['val_acc']),  max(history_s3_50['val_acc']),  max(history_s4_50['val_acc'])]
best_acc_100 = [max(history_s2_100['val_acc']), max(history_s3_100['val_acc']), max(history_s4_100['val_acc'])]
best_loss_50  = [min(history_s2_50['val_loss']),  min(history_s3_50['val_loss']),  min(history_s4_50['val_loss'])]
best_loss_100 = [min(history_s2_100['val_loss']), min(history_s3_100['val_loss']), min(history_s4_100['val_loss'])]
total_time_50  = [sum(history_s2_50['epoch_times']),  sum(history_s3_50['epoch_times']),  sum(history_s4_50['epoch_times'])]
total_time_100 = [sum(history_s2_100['epoch_times']), sum(history_s3_100['epoch_times']), sum(history_s4_100['epoch_times'])]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- 1. Best Validation Accuracy: 50 vs 100 ---
x = np.arange(len(student_names))
width = 0.35
bars1 = axes[0].bar(x - width/2, best_acc_50, width, label='50 Epochs',
                     color='#3498db', edgecolor='white', linewidth=1.5)
bars2 = axes[0].bar(x + width/2, best_acc_100, width, label='100 Epochs',
                     color='#e74c3c', edgecolor='white', linewidth=1.5)
for bar, val in zip(bars1, best_acc_50):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold', color='#3498db')
for bar, val in zip(bars2, best_acc_100):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold', color='#e74c3c')
axes[0].set_xticks(x)
axes[0].set_xticklabels(student_names, fontsize=10)
axes[0].set_ylabel('Best Validation Accuracy (%)', fontsize=12)
axes[0].set_title('Best Val Accuracy: 50 vs 100 Epochs', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# --- 2. Best Validation Loss: 50 vs 100 ---
bars3 = axes[1].bar(x - width/2, best_loss_50, width, label='50 Epochs',
                     color='#3498db', edgecolor='white', linewidth=1.5)
bars4 = axes[1].bar(x + width/2, best_loss_100, width, label='100 Epochs',
                     color='#e74c3c', edgecolor='white', linewidth=1.5)
for bar, val in zip(bars3, best_loss_50):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold', color='#3498db')
for bar, val in zip(bars4, best_loss_100):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold', color='#e74c3c')
axes[1].set_xticks(x)
axes[1].set_xticklabels(student_names, fontsize=10)
axes[1].set_ylabel('Best Validation Loss', fontsize=12)
axes[1].set_title('Best Val Loss: 50 vs 100 Epochs', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

# --- 3. Training Time: 50 vs 100 ---
bars5 = axes[2].bar(x - width/2, [t/60 for t in total_time_50], width, label='50 Epochs',
                     color='#3498db', edgecolor='white', linewidth=1.5)
bars6 = axes[2].bar(x + width/2, [t/60 for t in total_time_100], width, label='100 Epochs',
                     color='#e74c3c', edgecolor='white', linewidth=1.5)
for bar, val in zip(bars5, total_time_50):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{val/60:.1f}m', ha='center', fontsize=10, fontweight='bold', color='#3498db')
for bar, val in zip(bars6, total_time_100):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{val/60:.1f}m', ha='center', fontsize=10, fontweight='bold', color='#e74c3c')
axes[2].set_xticks(x)
axes[2].set_xticklabels(student_names, fontsize=10)
axes[2].set_ylabel('Training Time (minutes)', fontsize=12)
axes[2].set_title('Total Training Time: 50 vs 100 Epochs', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Final Comparison — 50 Epochs vs 100 Epochs', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# --- Print final comprehensive summary ---
print("\n" + "=" * 95)
print(f"{'COMPREHENSIVE RESULTS: 50 vs 100 EPOCHS':^95}")
print("=" * 95)
print(f"{'Student':<30} {'50ep Acc':<12} {'100ep Acc':<12} {'50ep Loss':<12} {'100ep Loss':<12} {'Improvement':<12}")
print("-" * 95)

students_data = [
    ('Student 2 (CNN)',            best_acc_50[0], best_acc_100[0], best_loss_50[0], best_loss_100[0]),
    ('Student 3 (CNN+BN)',        best_acc_50[1], best_acc_100[1], best_loss_50[1], best_loss_100[1]),
    ('Student 4 (CNN+BN+RMSProp)', best_acc_50[2], best_acc_100[2], best_loss_50[2], best_loss_100[2]),
]

for name, acc50, acc100, loss50, loss100 in students_data:
    improvement = acc100 - acc50
    arrow = '↑' if improvement > 0 else '↓' if improvement < 0 else '→'
    print(f"{name:<30} {acc50:<12.1f} {acc100:<12.1f} {loss50:<12.4f} {loss100:<12.4f} {arrow} {abs(improvement):.1f}%")

print("=" * 95)
print(f"\n★ Best overall: ", end="")
all_accs = best_acc_50 + best_acc_100
best_idx = np.argmax(all_accs)
epoch_label = '50 epochs' if best_idx < 3 else '100 epochs'
student_idx = best_idx % 3
student_labels = ['Student 2 (CNN)', 'Student 3 (CNN+BN)', 'Student 4 (CNN+BN+RMSProp)']
print(f"{student_labels[student_idx]} @ {epoch_label} with {all_accs[best_idx]:.1f}% validation accuracy")

# Mandatory Step for model 1 to 4 -  The Training Process
# Function to run the training for 50 epochs and 100 epochs.

In [ ]:
import time

# ============================================================
# TRAINING FUNCTION — with auto-plotting of learning curves
# ============================================================

def plot_training_history(history, title='Training History'):
    """
    Plots training & validation loss and accuracy curves for a single model.

    Args:
        history (dict): Must contain 'train_loss', 'val_loss', 'train_acc', 'val_acc'.
        title (str): Plot title (e.g., 'Model 1: FC Network').
    """
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Loss curves ---
    axes[0].plot(epochs, history['train_loss'], 'o-', color='#e74c3c', linewidth=2,
                 markersize=3, label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], 's-', color='#3498db', linewidth=2,
                 markersize=3, label='Val Loss')
    axes[0].fill_between(epochs, history['train_loss'], history['val_loss'],
                          alpha=0.1, color='#8e44ad')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{title} — Loss', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)

    # Mark best val loss
    best_val_epoch = np.argmin(history['val_loss']) + 1
    best_val_loss = min(history['val_loss'])
    axes[0].axvline(x=best_val_epoch, color='green', linestyle='--', alpha=0.5)
    axes[0].annotate(f'Best: {best_val_loss:.4f}\n(epoch {best_val_epoch})',
                     xy=(best_val_epoch, best_val_loss),
                     xytext=(best_val_epoch + len(epochs)*0.05, best_val_loss),
                     fontsize=9, color='green', fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='green'))

    # --- Accuracy curves ---
    axes[1].plot(epochs, history['train_acc'], 'o-', color='#e74c3c', linewidth=2,
                 markersize=3, label='Train Acc')
    axes[1].plot(epochs, history['val_acc'], 's-', color='#3498db', linewidth=2,
                 markersize=3, label='Val Acc')
    axes[1].fill_between(epochs, history['train_acc'], history['val_acc'],
                          alpha=0.1, color='#8e44ad')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title(f'{title} — Accuracy', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)

    # Mark best val accuracy
    best_acc_epoch = np.argmax(history['val_acc']) + 1
    best_acc = max(history['val_acc'])
    axes[1].axvline(x=best_acc_epoch, color='green', linestyle='--', alpha=0.5)
    axes[1].annotate(f'Best: {best_acc:.1f}%\n(epoch {best_acc_epoch})',
                     xy=(best_acc_epoch, best_acc),
                     xytext=(best_acc_epoch + len(epochs)*0.05, best_acc - 5),
                     fontsize=9, color='green', fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color='green'))

    plt.tight_layout()
    plt.show()


def plot_compare_histories(histories_dict):
    """
    Overlaid comparison of validation loss & accuracy for all models.

    Args:
        histories_dict (dict): Keys are Model names, values are history dicts.
            Example: {'Model 1 (FC)': history1, 'Model 2 (CNN)': history2, ...}
    """
    colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6']
    markers = ['o', 's', '^', 'D', 'v']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for idx, (name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history['val_loss']) + 1)
        color = colors[idx % len(colors)]
        marker = markers[idx % len(markers)]

        # Validation Loss
        axes[0].plot(epochs, history['val_loss'], f'{marker}-', color=color,
                     linewidth=2, markersize=4, label=name, alpha=0.85)

        # Validation Accuracy
        axes[1].plot(epochs, history['val_acc'], f'{marker}-', color=color,
                     linewidth=2, markersize=4, label=name, alpha=0.85)

    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Validation Loss', fontsize=12)
    axes[0].set_title('Validation Loss — All Models', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Validation Accuracy (%)', fontsize=12)
    axes[1].set_title('Validation Accuracy — All Models', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training Comparison Across All Models', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # --- Print final results summary ---
    print("\n" + "=" * 80)
    print(f"{'TRAINING RESULTS SUMMARY':^80}")
    print("=" * 80)
    print(f"{'Model':<30} {'Best Val Loss':<18} {'Best Val Acc':<18} {'Final Train Acc':<18}")
    print("-" * 80)
    for name, h in histories_dict.items():
        print(f"{name:<30} {min(h['val_loss']):<18.4f} {max(h['val_acc']):<18.1f} {h['train_acc'][-1]:<18.1f}")
    print("=" * 80)


def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=50, device='cpu', plot=True, title='Model'):
    """
    Train a PyTorch model and track metrics per epoch.

    Args:
        model (nn.Module): The neural network model to train.
        train_loader (DataLoader): Training data loader.
        val_loader (DataLoader): Validation data loader.
        criterion: Loss function.
        optimizer: Optimiser instance.
        num_epochs (int): Number of training epochs (default: 50).
        device (str): Device to train on ('cpu' or 'cuda').
        plot (bool): If True, auto-plot training curves after training.
        title (str): Title for the plot (e.g., 'Model 1: FC Network').

    Returns:
        dict: Training history with keys:
              'train_loss', 'val_loss', 'train_acc', 'val_acc', 'epoch_times'
    """
    model = model.to(device)

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'epoch_times': []
    }

    total_start = time.time()

    for epoch in range(num_epochs):
        epoch_start = time.time()

        # ---- Training phase ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total

        # ---- Validation phase ----
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_running_loss / val_total
        val_acc = 100.0 * val_correct / val_total

        epoch_time = time.time() - epoch_start

        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)

        # Print progress every 10 epochs + first and last
        if (epoch + 1) % 10 == 0 or epoch == 0 or (epoch + 1) == num_epochs:
            print(f"Epoch [{epoch+1:>3}/{num_epochs}]  "
                  f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:6.2f}%  |  "
                  f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:6.2f}%  "
                  f"[{epoch_time:.1f}s]")

    total_time = time.time() - total_start
    print(f"\n{'─'*60}")
    print(f"Training complete in {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"Best Val Accuracy: {max(history['val_acc']):.2f}% (epoch {np.argmax(history['val_acc'])+1})")
    print(f"Best Val Loss:     {min(history['val_loss']):.4f} (epoch {np.argmin(history['val_loss'])+1})")
    print(f"{'─'*60}")

    # Auto-plot training curves
    if plot:
        plot_training_history(history, title=title)

    return history


# Check for GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training device: {device}")
print(f"\ntrain_model() is ready.")
print("Usage: history = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=50)")
print("\nVisualization functions:")
print("  • plot_training_history(history, title)     — Individual Model curves")
print("  • plot_compare_histories(histories_dict)    — Overlaid comparison of all models")

## Training — 50 Epochs (All 4 Models)
> **Note**: Model 1 (FC) uses the CSV audio features, while Models 2–4 (CNNs) use MEL spectrogram images.  
> Model 1 is excluded from image-based training below since it requires tabular features, not images.

In [ ]:
# ============================================================
# TRAIN ALL CNN STUDENTS — 50 EPOCHS
# ============================================================
# NOTE: Model 1 (FC) expects tabular audio features (57-dim),
#       not images. It is excluded from image-based training here.
#       Models 2–4 all use MEL spectrogram images (3x180x180).

print("=" * 65)
print("  TRAINING — 50 EPOCHS")
print("=" * 65)

# --- Model 2: CNN (No BatchNorm, Adam) ---
print("\n▶ Model 2: CNN (No BatchNorm) — Adam optimiser")
print("─" * 50)
history_s2_50 = train_model(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_criterion,
    optimizer=cnn_optimizer,
    num_epochs=50,
    device=device,
    plot=True,
    title='Model 2: CNN (50 epochs)'
)

# --- Model 3: CNN + BatchNorm (Adam) ---
print("\n▶ Model 3: CNN + BatchNorm — Adam optimiser")
print("─" * 50)
history_s3_50 = train_model(
    model=cnn_bn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_bn_criterion,
    optimizer=cnn_bn_optimizer,
    num_epochs=50,
    device=device,
    plot=True,
    title='Model 3: CNN + BatchNorm (50 epochs)'
)

# --- Model 4: CNN + BatchNorm (RMSProp) ---
print("\n▶ Model 4: CNN + BatchNorm — RMSProp optimiser")
print("─" * 50)
history_s4_50 = train_model(
    model=cnn_bn_rms_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_bn_rms_criterion,
    optimizer=cnn_bn_rms_optimizer,
    num_epochs=50,
    device=device,
    plot=True,
    title='Model 4: CNN + BN + RMSProp (50 epochs)'
)

In [ ]:
# ============================================================
# VISUALIZATION: Compare All Models After 50 Epochs
# ============================================================

histories_50 = {
    'Model 2 (CNN)':            history_s2_50,
    'Model 3 (CNN+BN)':        history_s3_50,
    'Model 4 (CNN+BN+RMSProp)': history_s4_50,
}

plot_compare_histories(histories_50)

## Training — 100 Epochs (All 4 Models)
> Reinitialise all models with fresh weights and train for 100 epochs to observe longer-term convergence behaviour.

In [ ]:
# ============================================================
# TRAIN ALL CNN STUDENTS — 100 EPOCHS (Fresh Models)
# ============================================================
# Reinitialise models so training starts from scratch (not continuing from 50 epochs)

print("=" * 65)
print("  TRAINING — 100 EPOCHS (Fresh Models)")
print("=" * 65)

# --- Reinitialise Model 2: CNN (No BatchNorm, Adam) ---
cnn_model_100 = MusicGenreCNN(num_classes=10, dropout=0.4)
cnn_criterion_100 = nn.CrossEntropyLoss()
cnn_optimizer_100 = optim.Adam(cnn_model_100.parameters(), lr=1e-3)

print("\n▶ Model 2: CNN (No BatchNorm) — Adam optimiser — 100 epochs")
print("─" * 50)
history_s2_100 = train_model(
    model=cnn_model_100,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_criterion_100,
    optimizer=cnn_optimizer_100,
    num_epochs=100,
    device=device,
    plot=True,
    title='Model 2: CNN (100 epochs)'
)

# --- Reinitialise Model 3: CNN + BatchNorm (Adam) ---
cnn_bn_model_100 = MusicGenreCNN_BatchNorm(num_classes=10, dropout=0.4)
cnn_bn_criterion_100 = nn.CrossEntropyLoss()
cnn_bn_optimizer_100 = optim.Adam(cnn_bn_model_100.parameters(), lr=1e-3)

print("\n▶ Model 3: CNN + BatchNorm — Adam optimiser — 100 epochs")
print("─" * 50)
history_s3_100 = train_model(
    model=cnn_bn_model_100,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_bn_criterion_100,
    optimizer=cnn_bn_optimizer_100,
    num_epochs=100,
    device=device,
    plot=True,
    title='Model 3: CNN + BatchNorm (100 epochs)'
)

# --- Reinitialise Model 4: CNN + BatchNorm (RMSProp) ---
cnn_bn_rms_model_100 = MusicGenreCNN_BatchNorm_RMSProp(num_classes=10, dropout=0.4)
cnn_bn_rms_criterion_100 = nn.CrossEntropyLoss()
cnn_bn_rms_optimizer_100 = optim.RMSprop(
    cnn_bn_rms_model_100.parameters(),
    lr=1e-3, alpha=0.99, momentum=0.0, weight_decay=1e-4
)

print("\n▶ Model 4: CNN + BatchNorm — RMSProp optimiser — 100 epochs")
print("─" * 50)
history_s4_100 = train_model(
    model=cnn_bn_rms_model_100,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=cnn_bn_rms_criterion_100,
    optimizer=cnn_bn_rms_optimizer_100,
    num_epochs=100,
    device=device,
    plot=True,
    title='Model 4: CNN + BN + RMSProp (100 epochs)'
)

# Training with Early Stopping — Models 5 & 6
# An extended version of `train_model()` that monitors validation loss and stops
# training when it has not improved for a configurable number of epochs (patience).
# The best model weights (lowest validation loss) are automatically restored.

In [ ]:
# ============================================================
# TRAINING FUNCTION WITH EARLY STOPPING
# ============================================================

def train_model_early_stopping(model, train_loader, val_loader, criterion, optimizer,
                                max_epochs=200, patience=15, min_delta=1e-4,
                                device='cpu', plot=True, title='Model'):
    """
    Train a PyTorch model with early stopping based on validation loss.

    Stops training when val_loss has not improved by at least `min_delta`
    for `patience` consecutive epochs. Restores the best model weights.

    Args:
        model (nn.Module): The neural network to train.
        train_loader (DataLoader): Training data loader.
        val_loader (DataLoader): Validation data loader.
        criterion: Loss function.
        optimizer: Optimiser instance.
        max_epochs (int): Maximum number of epochs (upper bound).
        patience (int): Number of epochs with no improvement before stopping.
        min_delta (float): Minimum change in val_loss to count as improvement.
        device (str): 'cpu' or 'cuda'.
        plot (bool): Auto-plot training curves after training.
        title (str): Title for the plot.

    Returns:
        dict: Training history with keys:
              'train_loss', 'val_loss', 'train_acc', 'val_acc',
              'epoch_times', 'stopped_epoch', 'best_epoch'
    """
    import copy

    model = model.to(device)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [],  'val_acc': [],
        'epoch_times': [],
        'stopped_epoch': max_epochs,
        'best_epoch': 1,
    }

    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    total_start = time.time()

    for epoch in range(max_epochs):
        epoch_start = time.time()

        # ---- Training phase ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / total
        train_acc = 100.0 * correct / total

        # ---- Validation phase ----
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss = val_running_loss / val_total
        val_acc = 100.0 * val_correct / val_total
        epoch_time = time.time() - epoch_start

        # Record
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)

        # ---- Early Stopping Check ----
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            history['best_epoch'] = epoch + 1
        else:
            epochs_no_improve += 1

        # Print progress every 10 epochs, first, last, and when stopping
        if (epoch + 1) % 10 == 0 or epoch == 0:
            patience_bar = '#' * epochs_no_improve + '.' * (patience - epochs_no_improve)
            print(f"Epoch [{epoch+1:>3}/{max_epochs}]  "
                  f"Train Loss: {train_loss:.4f}  Acc: {train_acc:6.2f}%  |  "
                  f"Val Loss: {val_loss:.4f}  Acc: {val_acc:6.2f}%  "
                  f"[{epoch_time:.1f}s]  "
                  f"Patience [{patience_bar}]")

        if epochs_no_improve >= patience:
            history['stopped_epoch'] = epoch + 1
            print(f"\n>>> Early stopping triggered at epoch {epoch + 1} "
                  f"(no improvement for {patience} epochs)")
            print(f">>> Restoring best weights from epoch {history['best_epoch']} "
                  f"(val_loss={best_val_loss:.4f})")
            break

    # Restore best weights
    model.load_state_dict(best_model_weights)
    model.eval()

    total_time = time.time() - total_start
    actual_epochs = len(history['train_loss'])

    print(f"\n{'─'*65}")
    print(f"Training complete in {total_time:.1f}s ({total_time/60:.1f} min) — {actual_epochs} epochs")
    print(f"Best Val Loss:     {best_val_loss:.4f} (epoch {history['best_epoch']})")
    print(f"Best Val Accuracy: {max(history['val_acc']):.2f}% (epoch {np.argmax(history['val_acc'])+1})")
    if history['stopped_epoch'] < max_epochs:
        print(f"Early stopped:     epoch {history['stopped_epoch']} / {max_epochs}")
    else:
        print(f"Ran full {max_epochs} epochs (did not early stop)")
    print(f"{'─'*65}")

    if plot:
        plot_training_history(history, title=title)

    return history

print("train_model_early_stopping() is ready.")
print("  Stopping criterion: validation loss not improving for `patience` consecutive epochs")
print("  Best model weights are automatically restored after stopping.")
print("\nUsage:")
print("  history = train_model_early_stopping(model, train_loader, val_loader,")
print("               criterion, optimizer, max_epochs=200, patience=15, device=device)")


## Training — Model 5 (LSTM) with Early Stopping
> Train the bidirectional LSTM on the **original** audio Mel spectrogram dataset.
> Training stops automatically when validation loss converges (patience = 15 epochs).

In [ ]:
# ============================================================
# TRAIN MODEL 5 — LSTM with Early Stopping
# ============================================================

# Reinitialise Model 5 with fresh weights
lstm_model_es = MusicGenreLSTM(
    input_size=N_MELS,
    hidden_size=256,
    num_layers=2,
    num_classes=10,
    dropout=0.3,
    bidirectional=True
)
lstm_es_criterion = nn.CrossEntropyLoss()
lstm_es_optimizer = optim.Adam(lstm_model_es.parameters(), lr=1e-3)

print("=" * 65)
print("  MODEL 5 — Bidirectional LSTM (Original Audio Data)")
print("  Max epochs: 200  |  Patience: 15  |  Min delta: 1e-4")
print("=" * 65)

history_m5 = train_model_early_stopping(
    model=lstm_model_es,
    train_loader=audio_train_loader,
    val_loader=audio_val_loader,
    criterion=lstm_es_criterion,
    optimizer=lstm_es_optimizer,
    max_epochs=200,
    patience=15,
    min_delta=1e-4,
    device=device,
    plot=True,
    title='Model 5: LSTM (Early Stopping)'
)


## Training — Model 6 (LSTM + GAN Augmentation) with Early Stopping
> Train the same bidirectional LSTM architecture on the **augmented** dataset
> (real + GAN-generated Mel spectrograms). Same stopping criterion as Model 5
> to enable a fair comparison of how GAN augmentation affects convergence and accuracy.

In [ ]:
# ============================================================
# TRAIN MODEL 6 — LSTM + GAN Augmentation with Early Stopping
# ============================================================

# Reinitialise Model 6 with fresh weights (same architecture as Model 5)
lstm_gan_model_es = MusicGenreLSTM(
    input_size=N_MELS,
    hidden_size=256,
    num_layers=2,
    num_classes=10,
    dropout=0.3,
    bidirectional=True
)
lstm_gan_es_criterion = nn.CrossEntropyLoss()
lstm_gan_es_optimizer = optim.Adam(lstm_gan_model_es.parameters(), lr=1e-3)

print("=" * 65)
print("  MODEL 6 — Bidirectional LSTM + GAN Augmented Data")
print(f"  Training samples: {len(augmented_train_dataset)} (real + GAN)")
print("  Max epochs: 200  |  Patience: 15  |  Min delta: 1e-4")
print("=" * 65)

history_m6 = train_model_early_stopping(
    model=lstm_gan_model_es,
    train_loader=augmented_train_loader,    # ← augmented (real + GAN)
    val_loader=audio_val_loader,            # ← original val set (no augmentation)
    criterion=lstm_gan_es_criterion,
    optimizer=lstm_gan_es_optimizer,
    max_epochs=200,
    patience=15,
    min_delta=1e-4,
    device=device,
    plot=True,
    title='Model 6: LSTM + GAN Augmentation (Early Stopping)'
)


## Comparison: Model 5 vs Model 6 — Effect of GAN Augmentation

In [ ]:
# ============================================================
# COMPARISON: Model 5 vs Model 6 — Effect of GAN Augmentation
# ============================================================

histories_lstm = {
    'Model 5 (LSTM)':           history_m5,
    'Model 6 (LSTM + GAN aug)': history_m6,
}

# --- 1. Overlaid training curves ---
plot_compare_histories(histories_lstm)

# --- 2. Detailed side-by-side comparison ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

colors = {'m5': '#9b59b6', 'm6': '#e74c3c'}

# Top-left: Training loss
ax = axes[0, 0]
ax.plot(history_m5['train_loss'], '-', color=colors['m5'], linewidth=2, label='Model 5 Train')
ax.plot(history_m5['val_loss'], '--', color=colors['m5'], linewidth=2, label='Model 5 Val')
ax.plot(history_m6['train_loss'], '-', color=colors['m6'], linewidth=2, label='Model 6 Train')
ax.plot(history_m6['val_loss'], '--', color=colors['m6'], linewidth=2, label='Model 6 Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Top-right: Accuracy
ax = axes[0, 1]
ax.plot(history_m5['train_acc'], '-', color=colors['m5'], linewidth=2, label='Model 5 Train')
ax.plot(history_m5['val_acc'], '--', color=colors['m5'], linewidth=2, label='Model 5 Val')
ax.plot(history_m6['train_acc'], '-', color=colors['m6'], linewidth=2, label='Model 6 Train')
ax.plot(history_m6['val_acc'], '--', color=colors['m6'], linewidth=2, label='Model 6 Val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Curves', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom-left: Convergence bar chart
ax = axes[1, 0]
model_names = ['Model 5\n(LSTM)', 'Model 6\n(LSTM+GAN)']
stopped_epochs = [history_m5['stopped_epoch'], history_m6['stopped_epoch']]
best_epochs = [history_m5['best_epoch'], history_m6['best_epoch']]

x = np.arange(len(model_names))
width = 0.35
bars1 = ax.bar(x - width/2, stopped_epochs, width, label='Stopped Epoch',
               color=[colors['m5'], colors['m6']], alpha=0.7, edgecolor='black')
bars2 = ax.bar(x + width/2, best_epochs, width, label='Best Epoch',
               color=[colors['m5'], colors['m6']], alpha=1.0, edgecolor='black')

for bar, val in zip(list(bars1) + list(bars2), stopped_epochs + best_epochs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Epoch', fontsize=11)
ax.set_title('Convergence Speed', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()

# Bottom-right: Summary table
ax = axes[1, 1]
ax.axis('off')

table_data = [
    ['Metric', 'Model 5 (LSTM)', 'Model 6 (LSTM+GAN)'],
    ['Training Samples', f'{len(audio_splits["train"]["dataset"])}',
                         f'{len(augmented_train_dataset)}'],
    ['Best Val Loss', f'{min(history_m5["val_loss"]):.4f}',
                      f'{min(history_m6["val_loss"]):.4f}'],
    ['Best Val Accuracy', f'{max(history_m5["val_acc"]):.2f}%',
                          f'{max(history_m6["val_acc"]):.2f}%'],
    ['Best Epoch', str(history_m5['best_epoch']),
                   str(history_m6['best_epoch'])],
    ['Stopped Epoch', str(history_m5['stopped_epoch']),
                      str(history_m6['stopped_epoch'])],
    ['Total Train Time',
     f'{sum(history_m5["epoch_times"]):.1f}s',
     f'{sum(history_m6["epoch_times"]):.1f}s'],
    ['Epochs Trained', str(len(history_m5['train_loss'])),
                       str(len(history_m6['train_loss']))],
]

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.35, 0.3, 0.35])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

for j in range(3):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(table_data)):
    c = '#ecf0f1' if i % 2 == 0 else 'white'
    for j in range(3):
        table[i, j].set_facecolor(c)

# Highlight the winner row (best val acc)
best_m5 = max(history_m5['val_acc'])
best_m6 = max(history_m6['val_acc'])
winner_col = 1 if best_m5 >= best_m6 else 2
table[3, winner_col].set_facecolor('#d5f5e3')
table[3, winner_col].set_text_props(fontweight='bold')

plt.suptitle('Model 5 vs Model 6 — Impact of GAN Data Augmentation',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- Print final verdict ---
print("\n" + "=" * 65)
if best_m6 > best_m5:
    diff = best_m6 - best_m5
    print(f"  GAN augmentation IMPROVED accuracy by +{diff:.2f}%")
    print(f"  Model 6 ({best_m6:.2f}%) > Model 5 ({best_m5:.2f}%)")
elif best_m5 > best_m6:
    diff = best_m5 - best_m6
    print(f"  GAN augmentation did NOT improve accuracy (−{diff:.2f}%)")
    print(f"  Model 5 ({best_m5:.2f}%) > Model 6 ({best_m6:.2f}%)")
else:
    print(f"  Both models achieved identical accuracy: {best_m5:.2f}%")
print("=" * 65)


## Final Comparison Table — All 6 Models
> A comprehensive side-by-side comparison of architecture, parameters, training
> configuration, and performance across every model in this notebook.

In [ ]:
# ============================================================
# FINAL COMPARISON TABLE — ALL 6 MODELS
# ============================================================

# --- Gather all histories into a unified list ---
# Models 2-4 use the 100-epoch run; Models 5-6 use early-stopping run.
# Model 1 (FC) was not trained with images, so we check if its history exists.

all_models = []

# Helper to safely extract metrics from a history dict
def _metrics(h):
    return {
        'best_val_loss': min(h['val_loss']),
        'best_val_acc':  max(h['val_acc']),
        'final_train_acc': h['train_acc'][-1],
        'best_epoch_loss': int(np.argmin(h['val_loss'])) + 1,
        'best_epoch_acc':  int(np.argmax(h['val_acc'])) + 1,
        'epochs_trained':  len(h['train_loss']),
        'total_time':      sum(h['epoch_times']),
    }

# --- Model 1 ---
m1_params = sum(p.numel() for p in model.parameters())
all_models.append({
    'name': 'Model 1', 'desc': 'FC Network',
    'arch': 'FC (57→256→128→10)',
    'input': '57 audio features',
    'params': m1_params,
    'optimizer': 'Adam',
    'batchnorm': 'Yes (1d)',
    'dropout': '0.3',
    'data': 'CSV features',
    'history': None,  # not yet trained in notebook
})

# --- Model 2 ---
m2_params = sum(p.numel() for p in cnn_model.parameters())
all_models.append({
    'name': 'Model 2', 'desc': 'CNN',
    'arch': '4×Conv2d → 2×FC',
    'input': 'MEL images (3×180×180)',
    'params': m2_params,
    'optimizer': 'Adam',
    'batchnorm': 'No',
    'dropout': '0.4',
    'data': 'MEL images',
    'history': history_s2_100,
})

# --- Model 3 ---
m3_params = sum(p.numel() for p in cnn_bn_model.parameters())
all_models.append({
    'name': 'Model 3', 'desc': 'CNN + BatchNorm',
    'arch': '4×Conv2d + BN → 2×FC',
    'input': 'MEL images (3×180×180)',
    'params': m3_params,
    'optimizer': 'Adam',
    'batchnorm': 'Yes (2d+1d)',
    'dropout': '0.4',
    'data': 'MEL images',
    'history': history_s3_100,
})

# --- Model 4 ---
m4_params = sum(p.numel() for p in cnn_bn_rms_model.parameters())
all_models.append({
    'name': 'Model 4', 'desc': 'CNN + BN + RMSProp',
    'arch': '4×Conv2d + BN → 2×FC',
    'input': 'MEL images (3×180×180)',
    'params': m4_params,
    'optimizer': 'RMSProp',
    'batchnorm': 'Yes (2d+1d)',
    'dropout': '0.4',
    'data': 'MEL images',
    'history': history_m5 if False else history_s4_100,
})

# --- Model 5 ---
m5_params = sum(p.numel() for p in lstm_model_es.parameters())
all_models.append({
    'name': 'Model 5', 'desc': 'LSTM',
    'arch': 'Bi-LSTM (2L, 256h) → FC',
    'input': 'Mel spec (1×128×1292)',
    'params': m5_params,
    'optimizer': 'Adam',
    'batchnorm': 'Yes (1d)',
    'dropout': '0.3',
    'data': 'Audio WAV (original)',
    'history': history_m5,
})

# --- Model 6 ---
m6_params = sum(p.numel() for p in lstm_gan_model_es.parameters())
all_models.append({
    'name': 'Model 6', 'desc': 'LSTM + GAN Aug',
    'arch': 'Bi-LSTM (2L, 256h) → FC',
    'input': 'Mel spec (1×128×1292)',
    'params': m6_params,
    'optimizer': 'Adam',
    'batchnorm': 'Yes (1d)',
    'dropout': '0.3',
    'data': 'Audio WAV (real+GAN)',
    'history': history_m6,
})

# ============================================================
# 1. TEXT TABLE — printed in console
# ============================================================

print("=" * 130)
print(f"{'ALL 6 MODELS — COMPARISON TABLE':^130}")
print("=" * 130)

# Architecture table
print(f"\n{'ARCHITECTURE':^130}")
print("-" * 130)
print(f"{'':20s}", end='')
for m in all_models:
    print(f"{m['name']:18s}", end='')
print()
print("-" * 130)

for key, label in [('desc', 'Description'), ('arch', 'Architecture'),
                    ('input', 'Input'), ('data', 'Data Source'),
                    ('optimizer', 'Optimiser'), ('batchnorm', 'BatchNorm'),
                    ('dropout', 'Dropout')]:
    print(f"{label:20s}", end='')
    for m in all_models:
        print(f"{m[key]:18s}", end='')
    print()

print(f"{'Parameters':20s}", end='')
for m in all_models:
    print(f"{m['params']:>15,}   ", end='')
print()

# Performance table
print(f"\n{'TRAINING PERFORMANCE':^130}")
print("-" * 130)
print(f"{'':20s}", end='')
for m in all_models:
    print(f"{m['name']:18s}", end='')
print()
print("-" * 130)

for key, label, fmt in [
    ('epochs_trained', 'Epochs Trained', '{:d}'),
    ('best_val_loss', 'Best Val Loss', '{:.4f}'),
    ('best_val_acc', 'Best Val Acc (%)', '{:.2f}'),
    ('final_train_acc', 'Final Train Acc (%)', '{:.2f}'),
    ('best_epoch_acc', 'Best Epoch (Acc)', '{:d}'),
    ('total_time', 'Total Time (s)', '{:.1f}'),
]:
    print(f"{label:20s}", end='')
    for m in all_models:
        if m['history'] is not None:
            metrics = _metrics(m['history'])
            print(f"{fmt.format(metrics[key]):>15s}   ", end='')
        else:
            print(f"{'N/A':>15s}   ", end='')
    print()

print("=" * 130)

# Find best model by val accuracy
trained = [(m['name'], max(m['history']['val_acc'])) for m in all_models if m['history'] is not None]
best_name, best_acc = max(trained, key=lambda x: x[1])
print(f"\n  Best model by validation accuracy: {best_name} ({best_acc:.2f}%)")

# ============================================================
# 2. VISUAL TABLE (matplotlib) + Bar Charts
# ============================================================

fig = plt.figure(figsize=(22, 14))

# --- 2a. Grouped Bar Chart: Validation Accuracy ---
ax1 = fig.add_subplot(2, 2, 1)
names_short = [f"{m['name']}\n({m['desc']})" for m in all_models]
val_accs = []
for m in all_models:
    if m['history'] is not None:
        val_accs.append(max(m['history']['val_acc']))
    else:
        val_accs.append(0)

colors_all = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#1abc9c']
bars = ax1.bar(names_short, val_accs, color=colors_all, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, val_accs):
    if acc > 0:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    else:
        ax1.text(bar.get_x() + bar.get_width()/2., 1,
                 'N/A', ha='center', va='bottom', fontsize=10, color='gray')
ax1.set_ylabel('Best Validation Accuracy (%)', fontsize=11)
ax1.set_title('Best Validation Accuracy — All Models', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', labelsize=8)

# --- 2b. Parameter Count (Log Scale) ---
ax2 = fig.add_subplot(2, 2, 2)
param_vals = [m['params'] for m in all_models]
bars2 = ax2.bar(names_short, [np.log10(p) for p in param_vals],
                color=colors_all, edgecolor='white', linewidth=2)
for bar, p in zip(bars2, param_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.03,
             f'{p:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax2.set_ylabel('log₁₀(Parameters)', fontsize=11)
ax2.set_title('Total Parameters — All Models', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', labelsize=8)

# --- 2c. Training Time ---
ax3 = fig.add_subplot(2, 2, 3)
times = []
for m in all_models:
    if m['history'] is not None:
        times.append(sum(m['history']['epoch_times']))
    else:
        times.append(0)
bars3 = ax3.bar(names_short, times, color=colors_all, edgecolor='white', linewidth=2)
for bar, t in zip(bars3, times):
    if t > 0:
        lbl = f'{t:.0f}s' if t < 60 else f'{t/60:.1f}m'
        ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 lbl, ha='center', va='bottom', fontsize=10, fontweight='bold')
    else:
        ax3.text(bar.get_x() + bar.get_width()/2., 1,
                 'N/A', ha='center', va='bottom', fontsize=10, color='gray')
ax3.set_ylabel('Total Training Time (s)', fontsize=11)
ax3.set_title('Training Time — All Models', fontsize=13, fontweight='bold')
ax3.tick_params(axis='x', labelsize=8)

# --- 2d. Summary Table ---
ax4 = fig.add_subplot(2, 2, 4)
ax4.axis('off')

header = [''] + [m['name'] for m in all_models]
rows = []

rows.append(['Type'] + [m['desc'] for m in all_models])
rows.append(['Params'] + [f'{m["params"]:,}' for m in all_models])
rows.append(['Optimiser'] + [m['optimizer'] for m in all_models])
rows.append(['Dropout'] + [m['dropout'] for m in all_models])
rows.append(['Data'] + [m['data'] for m in all_models])

# Performance rows
perf_keys = [
    ('Epochs', 'epochs_trained', '{:d}'),
    ('Val Acc (%)', 'best_val_acc', '{:.2f}'),
    ('Val Loss', 'best_val_loss', '{:.4f}'),
    ('Train Time', 'total_time', None),
]
for label, key, fmt in perf_keys:
    row = [label]
    for m in all_models:
        if m['history'] is not None:
            metrics = _metrics(m['history'])
            if key == 'total_time':
                t = metrics[key]
                row.append(f'{t:.0f}s' if t < 60 else f'{t/60:.1f}m')
            else:
                row.append(fmt.format(metrics[key]))
        else:
            row.append('N/A')
    rows.append(row)

table_data = [header] + rows
table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                  colWidths=[0.14] + [0.13] * 6)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.5)

# Style header row
for j in range(7):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold', fontsize=8)

# Style row label column
for i in range(1, len(table_data)):
    table[i, 0].set_facecolor('#34495e')
    table[i, 0].set_text_props(color='white', fontweight='bold', fontsize=8)

# Alternate row colours
for i in range(1, len(table_data)):
    c = '#ecf0f1' if i % 2 == 0 else '#fdfefe'
    for j in range(1, 7):
        table[i, j].set_facecolor(c)

# Highlight best accuracy cell
if any(m['history'] is not None for m in all_models):
    best_col = 1 + val_accs.index(max(v for v in val_accs if v > 0))
    acc_row = next(i for i, r in enumerate(rows) if r[0] == 'Val Acc (%)') + 1
    table[acc_row, best_col].set_facecolor('#d5f5e3')
    table[acc_row, best_col].set_text_props(fontweight='bold', fontsize=9)

ax4.set_title('All Models — Summary', fontsize=13, fontweight='bold', pad=20)

plt.suptitle('Music Genre Classification — All 6 Models Comparison',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ============================================================
# 3. FINAL RANKING
# ============================================================
print("\n" + "=" * 65)
print(f"{'FINAL RANKING BY VALIDATION ACCURACY':^65}")
print("=" * 65)
ranked = sorted(trained, key=lambda x: x[1], reverse=True)
for rank, (name, acc) in enumerate(ranked, 1):
    medal = {1: '🥇', 2: '🥈', 3: '🥉'}.get(rank, '  ')
    print(f"  {medal} {rank}. {name:30s} {acc:.2f}%")
print("=" * 65)


## Evaluation on the Test Set — All 6 Models
> Run each trained model on its held-out **test set** (10% of data, never seen during training)
> and report accuracy, loss, a per-class classification report, and a confusion matrix.

In [ ]:
# ============================================================
# EVALUATION FUNCTION — Test Set Performance
# ============================================================

def evaluate_model(model, test_loader, criterion, class_names, device='cpu', title='Model'):
    """
    Evaluate a trained model on a test set.

    Returns a dict with test_loss, test_acc, all predictions, all true labels,
    and prints a classification report + confusion matrix.
    """
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = running_loss / total
    test_acc = 100.0 * correct / total

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # --- Print results ---
    print(f"\n{'='*65}")
    print(f"  {title} — TEST SET EVALUATION")
    print(f"{'='*65}")
    print(f"  Test Loss:     {test_loss:.4f}")
    print(f"  Test Accuracy: {test_acc:.2f}%  ({correct}/{total})")
    print(f"{'='*65}")

    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
                yticklabels=class_names, ax=axes[0], linewidths=0.5)
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('True', fontsize=11)
    axes[0].set_title(f'{title} — Confusion Matrix', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)

    # Normalised confusion matrix (percentages)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Oranges', xticklabels=class_names,
                yticklabels=class_names, ax=axes[1], linewidths=0.5, vmin=0, vmax=100)
    axes[1].set_xlabel('Predicted', fontsize=11)
    axes[1].set_ylabel('True', fontsize=11)
    axes[1].set_title(f'{title} — Normalised Confusion Matrix (%)', fontsize=13, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.show()

    return {
        'test_loss': test_loss,
        'test_acc': test_acc,
        'predictions': all_preds,
        'true_labels': all_labels,
        'confusion_matrix': cm,
    }

print("evaluate_model() is ready.")
print("Usage: results = evaluate_model(model, test_loader, criterion, class_names, device, title)")


### Test Evaluation — Models 2, 3, 4 (Image-Based CNNs)
> These models use the **MEL spectrogram image** test set (`test_loader`).

In [ ]:
# ============================================================
# TEST EVALUATION — Models 2, 3, 4 (Image-based CNNs)
# ============================================================
# Uses test_loader from the image dataset split (cell 8)
# Uses class_names from full_dataset.classes (cell 10)

image_class_names = full_dataset.classes
criterion_eval = nn.CrossEntropyLoss()

# --- Model 2: CNN ---
print("\n" + "#" * 65)
print("  MODEL 2: CNN (No BatchNorm)")
print("#" * 65)
results_m2 = evaluate_model(
    model=cnn_model_100, test_loader=test_loader,
    criterion=criterion_eval, class_names=image_class_names,
    device=device, title='Model 2: CNN'
)

# --- Model 3: CNN + BatchNorm ---
print("\n" + "#" * 65)
print("  MODEL 3: CNN + BatchNorm")
print("#" * 65)
results_m3 = evaluate_model(
    model=cnn_bn_model_100, test_loader=test_loader,
    criterion=criterion_eval, class_names=image_class_names,
    device=device, title='Model 3: CNN + BatchNorm'
)

# --- Model 4: CNN + BN + RMSProp ---
print("\n" + "#" * 65)
print("  MODEL 4: CNN + BatchNorm + RMSProp")
print("#" * 65)
results_m4 = evaluate_model(
    model=cnn_bn_rms_model_100, test_loader=test_loader,
    criterion=criterion_eval, class_names=image_class_names,
    device=device, title='Model 4: CNN + BN + RMSProp'
)


### Test Evaluation — Models 5, 6 (Audio LSTM)
> These models use the **Mel spectrogram audio** test set (`audio_test_loader`).

In [ ]:
# ============================================================
# TEST EVALUATION — Models 5, 6 (Audio LSTM)
# ============================================================
# Uses audio_test_loader from the audio dataset split (cell 12)
# Uses audio_label_encoder.classes_ for genre names

audio_class_names = list(audio_label_encoder.classes_)
criterion_eval = nn.CrossEntropyLoss()

# --- Model 5: LSTM ---
print("\n" + "#" * 65)
print("  MODEL 5: Bidirectional LSTM")
print("#" * 65)
results_m5 = evaluate_model(
    model=lstm_model_es, test_loader=audio_test_loader,
    criterion=criterion_eval, class_names=audio_class_names,
    device=device, title='Model 5: LSTM'
)

# --- Model 6: LSTM + GAN Augmentation ---
print("\n" + "#" * 65)
print("  MODEL 6: Bidirectional LSTM + GAN Augmentation")
print("#" * 65)
results_m6 = evaluate_model(
    model=lstm_gan_model_es, test_loader=audio_test_loader,
    criterion=criterion_eval, class_names=audio_class_names,
    device=device, title='Model 6: LSTM + GAN Aug'
)


### Test Set Results — All Models Summary

In [ ]:
# ============================================================
# TEST SET RESULTS — ALL MODELS SUMMARY
# ============================================================

all_results = {
    'Model 2 (CNN)':            results_m2,
    'Model 3 (CNN+BN)':        results_m3,
    'Model 4 (CNN+BN+RMSProp)':results_m4,
    'Model 5 (LSTM)':          results_m5,
    'Model 6 (LSTM+GAN)':      results_m6,
}

# --- 1. Text summary ---
print("=" * 75)
print(f"{'TEST SET RESULTS — ALL MODELS':^75}")
print("=" * 75)
print(f"{'Model':<30} {'Test Loss':<15} {'Test Accuracy':<15} {'Correct':<15}")
print("-" * 75)
for name, r in all_results.items():
    total = len(r['true_labels'])
    correct = int(r['test_acc'] * total / 100)
    print(f"{name:<30} {r['test_loss']:<15.4f} {r['test_acc']:<14.2f}% {correct}/{total}")
print("=" * 75)

# --- 2. Bar chart comparison ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

names = list(all_results.keys())
accs = [r['test_acc'] for r in all_results.values()]
losses = [r['test_loss'] for r in all_results.values()]
colors_bar = ['#e67e22', '#2ecc71', '#e74c3c', '#9b59b6', '#1abc9c']

# Accuracy bars
bars = axes[0].bar(range(len(names)), accs, color=colors_bar, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Test Accuracy — All Models', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, max(accs) + 8)

# Loss bars
bars2 = axes[1].bar(range(len(names)), losses, color=colors_bar, edgecolor='white', linewidth=2)
for bar, loss in zip(bars2, losses):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{loss:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=8)
axes[1].set_ylabel('Test Loss', fontsize=11)
axes[1].set_title('Test Loss — All Models', fontsize=13, fontweight='bold')

plt.suptitle('Test Set Performance — All Trained Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --- 3. Final ranking ---
ranked = sorted(all_results.items(), key=lambda x: x[1]['test_acc'], reverse=True)
print("\n" + "=" * 55)
print(f"{'FINAL TEST SET RANKING':^55}")
print("=" * 55)
for rank, (name, r) in enumerate(ranked, 1):
    medal = {1: ' 1st', 2: ' 2nd', 3: ' 3rd'}.get(rank, f' {rank}th')
    print(f"  {medal}  {name:<30s} {r['test_acc']:.2f}%")
print("=" * 55)

# --- 4. Validation vs Test accuracy comparison ---
print(f"\n{'VALIDATION vs TEST ACCURACY':^55}")
print("-" * 55)
print(f"{'Model':<30} {'Val Acc':<12} {'Test Acc':<12} {'Gap':<10}")
print("-" * 55)

val_hist = {
    'Model 2 (CNN)':             history_s2_100,
    'Model 3 (CNN+BN)':         history_s3_100,
    'Model 4 (CNN+BN+RMSProp)': history_s4_100,
    'Model 5 (LSTM)':           history_m5,
    'Model 6 (LSTM+GAN)':       history_m6,
}

for name in all_results:
    val_acc = max(val_hist[name]['val_acc'])
    test_acc = all_results[name]['test_acc']
    gap = test_acc - val_acc
    sign = '+' if gap >= 0 else ''
    print(f"  {name:<28s} {val_acc:>6.2f}%     {test_acc:>6.2f}%     {sign}{gap:.2f}%")
print("-" * 55)
